# The 🤗 datasets library
## [Introduction](https://huggingface.co/course/chapter5/1?fw=pt)

In [Chapter 3](https://huggingface.co/course/chapter3) you got your first taste of the 🤗 Datasets library and saw that there were three main steps when it came to fine-tuning a model:
- Load a dataset from the Hugging Face Hub.
- Preprocess the data with `Dataset.map()`.
- Load and compute metrics.

But this is just scratching the surface of what 🤗 Datasets can do! In this chapter, we will take a deep dive into the library. Along the way, we'll find answers to the following questions:
- What do you do when your dataset is not on the Hub?
- How can you slice and dice a dataset? (And what if you *really* need to use Pandas?)
- What do you do when your dataset is huge and will melt your laptop's RAM?
- What the heck are "memory mapping" and Apache Arrow?
- How can you create your own dataset and push it to the Hub?

The techniques you learn here will prepare you for the advanced tokenization and fine-tuning tasks in [Chapter 6](https://huggingface.co/course/chapter6) and [Chapter 7](https://huggingface.co/course/chapter7) — so grab a coffee and let's get started!

## [What if my dataset isn't on the Hub?](https://huggingface.co/course/chapter5/2?fw=pt)

You know how to use the Hugging Face Hub to download datasets, but you'll often find yourself working with data that is stored either on your laptop or on a remote server. In this section we'll show you how 🤗 Datasets can be used to load datasets that aren't available on the Hugging Face Hub.

In [1]:
from IPython.display import HTML
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/HyQgpJTkRdE" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Working with local and remote datasets
🤗 Datasets provides loading scripts to handle the loading of local and remote datasets. It supports several common data formats, such as:

| Data format | Loading script | Example
| :---- | :------- | :---- |
| CSV & TSV | `csv` | `load_dataset("csv", data_files="my_file.csv")` |
| Text files | `text` | `load_dataset("text", data_files="my_file.txt")` |
| JSON & JSON Lines | `json` | `load_dataset("json", data_files="my_file.jsonl")` |
| Pickled DataFrames | `pandas` | `load_dataset("pandas", data_files="my_dataframe.pkl")` |

As shown in the table, for each data format we just need to specify the type of loading script in the `load_dataset()` function, along with a `data_files` argument that specifies the path to one or more files. Let's start by loading a dataset from local files; later we'll see how to do the same with remote files.

### Loading a local dataset

For this example we'll use the [SQuAD-it](https://github.com/crux82/squad-it/) dataset, which is a large-scale dataset for question answering in Italian.

The training and test splits are hosted on GitHub, so we can download them with a simple `wget` command:

In [2]:
# the following commands to download the data files and move them to the "data" folder need to run only once
#!wget "https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz"
#!wget "https://github.com/crux82/squad-it/raw/master/SQuAD_it-test.json.gz"
#!mv "SQuAD_it-train.json.gz" "sections/section_5/data"
#!mv "SQuAD_it-test.json.gz" "sections/section_5/data"

This will download two compressed files called *SQuAD_it-train.json.gz* and *SQuAD_it-test.json.gz*, which we can decompress with the Linux `gzip` command:

In [3]:
# the following command needs to run only once
#!gzip -dkv sections/section_5/data/SQuAD_it-*.json.gz

We can see that the compressed files have been replaced with *SQuAD_it-train.json* and *SQuAD_it-test.json*, and that the data is stored in the JSON format.
> <font color="darkgreen">✎ If you're wondering why there's a `!` character in the above shell commands, that's because we're running them within a Jupyter notebook. Simply remove the prefix if you want to download and unzip the dataset within a terminal.</font>

To load a JSON file with the `load_dataset()` function, we just need to know if we're dealing with ordinary JSON (similar to a nested dictionary) or JSON Lines (line-separated JSON). Like many question answering datasets, SQuAD-it uses the nested format, with all the text stored in a `data` field. This means we can load the dataset by specifying the `field` argument as follows:

In [4]:
from datasets import load_dataset
squad_it_dataset = load_dataset("json", data_files="sections/section_5/data/SQuAD_it-train.json", field="data")

By default, loading local files creates a `DatasetDict` object with a `train` split. We can see this by inspecting the `squad_it_dataset` object:

In [5]:
squad_it_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 442
    })
})

This shows us the number of rows and the column names associated with the training set. We can view one of the examples by indexing into the `train` split as follows:

In [6]:
print(squad_it_dataset["train"][0]["title"])
for paragraph in squad_it_dataset["train"][0]["paragraphs"][:2]:
    print(f"\n{paragraph}")

Terremoto del Sichuan del 2008

{'context': "Il terremoto del Sichuan del 2008 o il terremoto del Gran Sichuan, misurato a 8.0 Ms e 7.9 Mw, e si è verificato alle 02:28:01 PM China Standard Time all' epicentro (06:28:01 UTC) il 12 maggio nella provincia del Sichuan, ha ucciso 69.197 persone e lasciato 18.222 dispersi.", 'qas': [{'answers': [{'answer_start': 29, 'text': '2008'}], 'id': '56cdca7862d2951400fa6826', 'question': 'In quale anno si è verificato il terremoto nel Sichuan?'}, {'answers': [{'answer_start': 232, 'text': '69.197'}], 'id': '56cdca7862d2951400fa6828', 'question': 'Quante persone sono state uccise come risultato?'}, {'answers': [{'answer_start': 29, 'text': '2008'}], 'id': '56d4f9902ccc5a1400d833c0', 'question': 'Quale anno ha avuto luogo il terremoto del Sichuan?'}, {'answers': [{'answer_start': 78, 'text': '8.0 Ms e 7.9 Mw'}], 'id': '56d4f9902ccc5a1400d833c1', 'question': 'Che cosa ha fatto la misura di sisma?'}, {'answers': [{'answer_start': 183, 'text': '12 maggio

Great, we've loaded our first local dataset! But while this worked for the training set, what we really want is to include both the `train` and `test` splits in a single `DatasetDict` object so we can apply `Dataset.map()` functions across both splits at once. To do this, we can provide a dictionary to the `data_files` argument that maps each split name to a file associated with that split:

In [7]:
data_files = {"train": "sections/section_5/data/SQuAD_it-train.json", "test": "sections/section_5/data/SQuAD_it-test.json"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
squad_it_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 442
    })
    test: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 48
    })
})

This is exactly what we wanted. Now, we can apply various preprocessing techniques to clean up the data, tokenize the reviews, and so on.

> The `data_files` argument of the `load_dataset()` function is quite flexible and can be either a single file path, a list of file paths, or a dictionary that maps split names to file paths. You can also glob files that match a specified pattern according to the rules used by the Unix shell (e.g., you can glob all the JSON files in a directory as a single split by setting `data_files="*.json"`). See the 🤗 Datasets [documentation](https://huggingface.co/docs/datasets/loading.html#local-and-remote-files) for more details.

The loading scripts in 🤗 Datasets actually support automatic decompression of the input files, so we could have skipped the use of `gzip` by pointing the `data_files` argument directly to the compressed files:

In [8]:
data_files = {"train": "sections/section_5/data/SQuAD_it-train.json.gz", "test": "sections/section_5/data/SQuAD_it-test.json.gz"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
squad_it_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 442
    })
    test: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 48
    })
})

This can be useful if you don't want to manually decompress many GZIP files. The automatic decompression also applies to other common formats like ZIP and TAR, so you just need to point `data_files` to the compressed files and you're good to go!

Now that you know how to load local files on your laptop or desktop, let's take a look at loading remote files.
### Loading a remote dataset

If you're working as a data scientist or coder in a company, there's a good chance the datasets you want to analyze are stored on some remote server. Fortunately, loading remote files is just as simple as loading local ones! Instead of providing a path to local files, we point the `data_files` argument of `load_dataset()` to one or more URLs where the remote files are stored. For example, for the SQuAD-it dataset hosted on GitHub, we can just point `data_files` to the <i>SQuAD_it-\*.json.gz</i> URLs as follows:

In [9]:
url = "https://github.com/crux82/squad-it/raw/master/"
data_files = {
    "train": url + "SQuAD_it-train.json.gz",
    "test": url + "SQuAD_it-test.json.gz",
}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
squad_it_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 442
    })
    test: Dataset({
        features: ['title', 'paragraphs'],
        num_rows: 48
    })
})

This returns the same `DatasetDict` object obtained above, but saves us the step of manually downloading and decompressing the <i>SQuAD_it-\*.json.gz</i> files. This wraps up our foray into the various ways to load datasets that aren't hosted on the Hugging Face Hub. Now that we've got a dataset to play with, let's get our hands dirty with various data-wrangling techniques!

> ✏️ Try it out! <font color="darkgreen">Pick another dataset hosted on GitHub or the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/index.php) and try loading it both locally and remotely using the techniques introduced above. For bonus points, try loading a dataset that's stored in a CSV or text format (see the [documentation](https://huggingface.co/docs/datasets/loading.html#local-and-remote-files) for more information on these formats).</font>

In [10]:
# Trying it out
## load dataset locally, from csv
!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00611/accelerometer.csv"
!mv "accelerometer.csv" "sections/section_5/data"
accelerometer_dataset = load_dataset("csv", data_files="sections/section_5/data/accelerometer.csv")
print(f"Accelerometer dataset:\n{accelerometer_dataset}")
## load dataset remotely
data_files = {"train": "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"}
iris_dataset = load_dataset("csv", data_files=data_files)
text = "Add some preprocessing to correct the `features` and to add the first instance (=current `features`)!"
print(f"Iris dataset:\n{iris_dataset}\n{text}")

--2025-03-24 15:01:30--  https://archive.ics.uci.edu/ml/machine-learning-databases/00611/accelerometer.csv
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
connected. to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... 
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘accelerometer.csv’

accelerometer.csv       [      <=>           ]   3.56M  2.54MB/s    in 1.4s    

2025-03-24 15:01:32 (2.54 MB/s) - ‘accelerometer.csv’ saved [3731094]



Generating train split: 0 examples [00:00, ? examples/s]

Accelerometer dataset:
DatasetDict({
    train: Dataset({
        features: ['wconfid', 'pctid', 'x', 'y', 'z'],
        num_rows: 153000
    })
})
Iris dataset:
DatasetDict({
    train: Dataset({
        features: ['5.1', '3.5', '1.4', '0.2', 'Iris-setosa'],
        num_rows: 149
    })
})
Add some preprocessing to correct the `features` and to add the first instance (=current `features`)!


## [Time to slice and dice](https://huggingface.co/course/chapter5/3?fw=pt)

Most of the time, the data you work with won't be perfectly prepared for training models. In this section we'll explore the various features that 🤗 Datasets provides to clean up your datasets.

In [11]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/tqfSFcPMgOI" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Slicing and dicing our data

Similar to Pandas, 🤗 Datasets provides several functions to manipulate the contents of `Dataset` and `DatasetDict` objects. We already encountered the `Dataset.map()` method in Chapter 3, and in this section we'll explore some of the other functions at our disposal.

For this example we'll use the [Drug Review Dataset](https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29) that's hosted on the [UC Irvine Machine Learning Repository}(https://archive.ics.uci.edu/ml/index.php), which contains patient reviews on various drugs, along with the condition being treated and a 10-star rating of the patient's satisfaction.

First we need to download and extract the data, which can be done with the `wget` and `unzip` commands:

In [12]:
# the following commands need to run only once
!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
!mv "drugsCom_raw.zip" "sections/section_5/data"
!unzip -o "sections/section_5/data/drugsCom_raw.zip" -d "sections/section_5/data"

--2025-03-24 15:01:40--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
connected. to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... 
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip’

drugsCom_raw.zip        [             <=>    ]  41.00M  12.0MB/s    in 4.5s    

2025-03-24 15:01:45 (9.12 MB/s) - ‘drugsCom_raw.zip’ saved [42989872]

Archive:  sections/section_5/data/drugsCom_raw.zip
  inflating: sections/section_5/data/drugsComTest_raw.tsv  
  inflating: sections/section_5/data/drugsComTrain_raw.tsv  


Since TSV is just a variant of CSV that uses tabs instead of commas as the separator, we can load these files by using the csv loading script and specifying the `delimiter` argument in the `load_dataset()` function as follows:

In [13]:
data_files = {"train": "sections/section_5/data/drugsComTrain_raw.tsv", "test": "sections/section_5/data/drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

A good practice when doing any sort of data analysis is to grab a small random sample to get a quick feel for the type of data you're working with. In 🤗 Datasets, we can create a random sample by chaining the `Dataset.shuffle()` and `Dataset.select()` functions together:

In [14]:
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
# Peek at the first few examples
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

Note that we've fixed the seed in `Dataset.shuffle()` for reproducibility purposes. `Dataset.select()` expects an iterable of indices, so we've passed `range(1000)` to grab the first 1,000 examples from the shuffled dataset. From this sample we can already see a few quirks in our dataset:
- The `Unnamed: 0` column looks suspiciously like an anonymized ID for each patient.
- The `condition` column includes a mix of uppercase and lowercase labels.
- The `reviews` are of varying length and contain a mix of Python line separators (`\r\n`) as well as HTML character codes like `&\#039;`.

Let's see how we can use 🤗 Datasets to deal with each of these issues. To test the patient ID hypothesis for the `Unnamed: 0` column, we can use the `Dataset.unique()` function to verify that the number of IDs matches the number of rows in each split:

In [15]:
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

This seems to confirm our hypothesis, so let's clean up the dataset a bit by renaming the `Unnamed: 0` column to something a bit more interpretable. We can use the `DatasetDict.rename_column()` function to rename the column across both splits in one go:

In [16]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

> ✏️ Try it out! <font color="darkgreen">Use the `Dataset.unique()` function to find the number of unique drugs and conditions in the training and test sets.</font>

In [17]:
# Trying it out
print(f'unique drugs in the "train" set:\t{len(drug_dataset["train"].unique("drugName"))}')
print(f'unique drugs in the "test" set: \t{len(drug_dataset["test"].unique("drugName"))}')
print(f'unique conditions in the "train" set:\t{len(drug_dataset["train"].unique("condition"))}')
print(f'unique conditions in the "test" set:\t{len(drug_dataset["test"].unique("condition"))}')

unique drugs in the "train" set:	3436
unique drugs in the "test" set: 	2637
unique conditions in the "train" set:	885
unique conditions in the "test" set:	709


Next, let's normalize all the `condition` labels using `Dataset.map()`. As we did with tokenization in [Chapter 3](https://huggingface.co/course/chapter3), we can define a simple function that can be applied across all the rows of each split in `drug_dataset`:

```python
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}
drug_dataset.map(lowercase_condition)

AttributeError: 'NoneType' object has no attribute 'lower'
```

Oh no, we've run into a problem with our map function! From the error we can infer that some of the entries in the `condition` column are `None`, which cannot be lowercased as they're not strings. Let's drop these rows using `Dataset.filter()`, which works in a similar way to `Dataset.map()` and expects a function that receives a single example of the dataset. Instead of writing an explicit function like:
```python
def filter_nones(x):
    return x["condition"] is not None
```
and then running `drug_dataset.filter(filter_nones)`, we can do this in one line using a *lambda function*. In Python, lambda functions are small functions that you can define without explicitly naming them. They take the general form:
```python
lambda <arguments> : <expression>
```
where `lambda` is one of Python's special [keywords](https://docs.python.org/3/reference/lexical_analysis.html#keywords), `<arguments>` is a list/set of comma-separated values that define the inputs to the function, and `<expression>` represents the operations you wish to execute. For example, we can define a simple lambda function that squares a number as follows:

In [18]:
lambda x : x * x

<function __main__.<lambda>(x)>

To apply this function to an input, we need to wrap it and the input in parentheses:

In [19]:
(lambda x: x * x)(3)

9

Similarly, we can define lambda functions with multiple arguments by separating them with commas. For example, we can compute the area of a triangle as follows:

In [20]:
(lambda base, height: 0.5 * base * height)(4, 8)

16.0

Lambda functions are handy when you want to define small, single-use functions (for more information about them, we recommend reading the excellent [Real Python tutorial](https://realpython.com/python-lambda/) by Andre Burgaud). In the 🤗 Datasets context, we can use lambda functions to define simple map and filter operations, so let's use this trick to eliminate the `None` entries in our dataset:

In [21]:
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

With the `None` entries removed, we can normalize our `condition` column:

In [22]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}

drug_dataset = drug_dataset.map(lowercase_condition)
# Check that lowercasing worked
drug_dataset["train"]["condition"][:3]

['left ventricular dysfunction', 'adhd', 'birth control']

It works! Now that we've cleaned up the labels, let's take a look at cleaning up the reviews themselves.

### Creating new columns
Whenever you're dealing with customer reviews, a good practice is to check the number of words in each review. A review might be just a single word like "Great!" or a full-blown essay with thousands of words, and depending on the use case you'll need to handle these extremes differently. To compute the number of words in each review, we'll use a rough heuristic based on splitting each text by whitespace.

Let's define a simple function that counts the number of words in each review:

In [23]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

Unlike our `lowercase_condition()` function, `compute_review_length()` returns a dictionary whose key does not correspond to one of the column names in the dataset. In this case, when `compute_review_length()` is passed to `Dataset.map()`, it will be applied to all the rows in the dataset to create a new `review_length` column:

In [24]:
drug_dataset = drug_dataset.map(compute_review_length)
# Inspect the first training example
drug_dataset["train"][0]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

As expected, we can see a `review_length` column has been added to our training set. We can sort this new column with `Dataset.sort()` to see what the extreme values look like:

In [25]:
drug_dataset["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['hepatitis c', 'adhd', 'birth control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

As we suspected, some reviews contain just a single word, which, although it may be okay for sentiment analysis, would not be informative if we want to predict the condition.

> <font color="darkgreen">🙋 An alternative way to add new columns to a dataset is with the `Dataset.add_column()` function. This allows you to provide the column as a Python list or NumPy array and can be handy in situations where `Dataset.map()` is not well suited for your analysis.</font>

Let's use the `Dataset.filter()` function to remove reviews that contain fewer than 30 words. Similarly to what we did with the condition column, we can filter out the very short reviews by requiring that the reviews have a length above this threshold:

In [26]:
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)

{'train': 138514, 'test': 46108}


As you can see, this has removed around 15% of the reviews from our original training and test sets.

> ✏️ Try it out! <font color="darkgreen">Use the `Dataset.sort()` function to inspect the reviews with the largest numbers of words. See the documentation to see which argument you need to use to sort the reviews by length in descending order.</font>

In [27]:
# Trying it out
drug_dataset_sorted_by_review_length = drug_dataset["train"].sort("review_length", reverse=True)
print(drug_dataset_sorted_by_review_length)
for i in range(3):
    review = drug_dataset_sorted_by_review_length["review"][i]
    review_length = drug_dataset_sorted_by_review_length["review_length"][i]
    print(f"\nreview {i}\nlength {review_length}\n{review[:500]}...")
# documentation: https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.sort

Dataset({
    features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
    num_rows: 138514
})

review 0
length 1894
"Two and a half months ago I was prescribed Venlafaxine to help prevent chronic migraines.
It did help the migraines (reduced them by almost half), but with it came a host of side effects that were far worse than the problem I was trying to get rid of.
Having now come off of the stuff, I would not recommend anyone ever use Venlafaxine unless they suffer from extreme / suicidal depression. I mean extreme in the most emphatic sense of the word. 
Before trying Venlafaxine, I was a writer. While ...

review 1
length 1162
"I don&rsquo;t find a lot of positive stories about antidepressants, or I find stories where people are taking the antidepressant the wrong way.

I wanted to share my experience.  A positive one.

I&rsquo;ve had generalized anxiety disorder, SEVERE OCD, and panic disorder for as long as I can remember.  M

The last thing we need to deal with is the presence of HTML character codes in our reviews. We can use Python's `html`
 module to unescape these characters, like so:

In [28]:
import html
text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

We'll use `Dataset.map()` to unescape all the HTML characters in our corpus:

In [29]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

As you can see, the `Dataset.map()` method is quite useful for processing data — and we haven't even scratched the surface of everything it can do!

### The `map()` method's superpowers

The `Dataset.map()` method takes a batched argument that, if set to `True`, causes it to send a batch of examples to the map function at once (the batch size is configurable but defaults to 1,000). For instance, the previous map function that unescaped all the HTML took a bit of time to run (you can read the time taken from the progress bars). We can speed this up by processing several elements at the same time using a list comprehension.

When you specify `batched=True` the function receives a dictionary with the fields of the dataset, but each value is now a *list of values*, and not just a single value. The return value of `Dataset.map()` should be the same: a dictionary with the fields we want to update or add to our dataset, and a list of values. For example, here is another way to unescape all HTML characters, but using `batched=True`:

In [30]:
new_drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]},
    batched=True
)
new_drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

If you're running this code in a notebook, you'll see that this command executes way faster than the previous one. And it's not because our reviews have already been HTML-unescaped — if you re-execute the instruction from the previous section (without `batched=True`), it will take the same amount of time as before. This is because list comprehensions are usually faster than executing the same code in a `for` loop, and we also gain some performance by accessing lots of elements at the same time instead of one by one.

Using `Dataset.map()` with `batched=True` will be essential to unlock the speed of the "fast" tokenizers that we'll encounter in [Chapter 6](https://huggingface.co/course/chapter6), which can quickly tokenize big lists of texts. For instance, to tokenize all the drug reviews with a fast tokenizer, we could use a function like this:

In [31]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)

As you saw in [Chapter 3](https://huggingface.co/course/chapter3), we can pass one or several examples to the tokenizer, so we can use this function with or without `batched=True`. Let's take this opportunity to compare the performance of the different options. In a notebook, you can time a one-line instruction by adding `%time` before the line of code you wish to measure:

In [32]:
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)

CPU times: user 12.5 ms, sys: 7.96 ms, total: 20.5 ms
Wall time: 19.5 ms


You can also time a whole cell by putting `%%time` at the beginning of the cell. On the hardware we executed this on, it showed 10.8s for this instruction (it's the number written after "Wall time").
> ✏️ Try it out! <font color="darkgreen">Execute the same instruction with and without `batched=True`, then try it with a slow tokenizer (add `use_fast=False` in the `AutoTokenizer.from_pretrained()` method) so you can see what numbers you get on your hardware.</font>

In [33]:
# Trying it out
## a fast tokenizer
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False)
## not a fast tokenizer
not_a_fast_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
def not_a_fast_tokenize_function(examples):
    return not_a_fast_tokenizer(examples["review"], truncation=True)
%time tokenized_dataset = drug_dataset.map(not_a_fast_tokenize_function, batched=True)
%time tokenized_dataset = drug_dataset.map(not_a_fast_tokenize_function, batched=False)

CPU times: user 27.1 ms, sys: 1.91 ms, total: 29 ms
Wall time: 28.6 ms
CPU times: user 14.2 ms, sys: 6.02 ms, total: 20.2 ms
Wall time: 20.2 ms
CPU times: user 313 ms, sys: 7.82 ms, total: 321 ms
Wall time: 324 ms
CPU times: user 305 ms, sys: 8.83 ms, total: 314 ms
Wall time: 314 ms


Here are the results we obtained with and without batching, with a fast and a slow tokenizer:

|Options|Fast tokenizer|Slow tokenizer|
|-------|--------------|--------------|
|`batched=True`|18.3ms|325s|
|`batched=False`|15.5ms|353ms|

This means that using a fast tokenizer with the `batched=True` option is (not) 30 times faster than its slow counterpart with no batching — this is truly amazing! That's the main reason why fast tokenizers are the default when using `AutoTokenizer` (and why they are called "fast"). They're able to achieve such a speedup because behind the scenes the tokenization code is executed in Rust, which is a language that makes it easy to parallelize code execution.

Parallelization is also the reason for the nearly 6x speedup the fast tokenizer achieves with batching: you can't parallelize a single tokenization operation, but when you want to tokenize lots of texts at the same time you can just split the execution across several processes, each responsible for its own texts.

`Dataset.map()` also has some parallelization capabilities of its own. Since they are not backed by Rust, they won't let a slow tokenizer catch up with a fast one, but they can still be helpful (especially if you're using a tokenizer that doesn't have a fast version). To enable multiprocessing, use the `num_proc` argument and specify the number of processes to use in your call to `Dataset.map()`:

In [34]:
slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
def slow_tokenize_function(examples):
    return slow_tokenizer(examples["review"], truncation=True)
tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 46108
    })
})

You can experiment a little with timing to determine the optimal number of processes to use; in our case 8 seemed to produce the best speed gain. Here are the numbers we got with and without multiprocessing:

|Options|Fast tokenizer|Slow tokenizer|
|-------|--------------|--------------|
|`batched=True`|13.6s|2min5s|
|`batched=False`|1min6s|2min34s|
|`batched=True`, `num_proc=8`|6.52s|41.3s|
|`batched=False`, `num_proc=8`|9.49s|45.2s|

Those are much more reasonable results for the slow tokenizer, but the performance of the fast tokenizer was also substantially improved. Note, however, that won't always be the case — for values of `num_proc` other than 8, our tests showed that it was faster to use `batched=True` without that option. In general, we don't recommend using Python multiprocessing for fast tokenizers with `batched=True`.
> <font color="darkgreen">Using `num_proc` to speed up your processing is usually a great idea, as long as the function you are using is not already doing some kind of multiprocessing of its own.</font>

All of this functionality condensed into a single method is already pretty amazing, but there's more! With `Dataset.map()` and `batched=True` you can change the number of elements in your dataset. This is super useful in many situations where you want to create several training features from one example, and we will need to do this as part of the preprocessing for several of the NLP tasks we'll undertake in [Chapter 7](https://huggingface.co/course/chapter7).
> <font color="darkgreen">💡 In machine learning, an *example* is usually defined as the set of *features* that we feed to the model. In some contexts, these features will be the set of columns in a `Dataset`, but in others (like here and for question answering), multiple features can be extracted from a single example and belong to a single column.</font>

Let's have a look at how it works! Here we will tokenize our examples and truncate them to a maximum length of 128, but we will ask the tokenizer to return *all* the chunks of the texts instead of just the first one. This can be done with `return_overflowing_tokens=True`:

In [35]:
def tokenize_and_split(examples):
    return tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
#
tokenize_and_split

<function __main__.tokenize_and_split(examples)>

Let's test this on one example before using `Dataset.map()` on the whole dataset:

In [36]:
result = tokenize_and_split(drug_dataset["train"][0])
[len(inp) for inp in result["input_ids"]]

[128, 49]

So, our first example in the training set became two features because it was tokenized to more than the maximum number of tokens we specified: the first one of length 128 and the second one of length 49. Now let's do this for all elements of the dataset!
```python
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)

ArrowInvalid: Column 1 named condition expected length 1463 but got length 1000
```

Oh no! That didn't work! Why not? Looking at the error message will give us a clue: there is a mismatch in the lengths of one of the columns, one being of length 1,463 and the other of length 1,000. If you've looked at the `Dataset.map()` documentation, you may recall that it's the number of samples passed to the function that we are mapping; here those 1,000 examples gave 1,463 new features, resulting in a shape error.

The problem is that we're trying to mix two different datasets of different sizes: the `drug_dataset` columns will have a certain number of examples (the 1,000 in our error), but the `tokenized_dataset` we are building will have more (the 1,463 in the error message). That doesn't work for a `Dataset`, so we need to either remove the columns from the old dataset or make them the same size as they are in the new dataset. We can do the former with the `remove_columns` argument:

In [37]:
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 68876
    })
})

Now this works without error. We can check that our new dataset has many more elements than the original dataset by comparing the lengths:

In [38]:
len(tokenized_dataset["train"]), len(drug_dataset["train"])

(206772, 138514)

We mentioned that we can also deal with the mismatched length problem by making the old columns the same size as the new ones. To do this, we will need the `overflow_to_sample_mapping` field the tokenizer returns when we set `return_overflowing_tokens=True`. It gives us a mapping from a new feature index to the index of the sample it originated from. Using this, we can associate each key present in our original dataset with a list of values of the right size by repeating the values of each example as many times as it generates new features:

In [39]:
def tokenize_and_split(examples):
    result = tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

We can see it works with `Dataset.map()` without us needing to remove the old columns:

In [40]:
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 68876
    })
})

We get the same number of training features as before, but here we've kept all the old fields. If you need them for some post-processing after applying your model, you might want to use this approach.

You've now seen how 🤗 Datasets can be used to preprocess a dataset in various ways. Although the processing functions of 🤗 Datasets will cover most of your model training needs, there may be times when you'll need to switch to Pandas to access more powerful features, like `DataFrame.groupby()` or high-level APIs for visualization. Fortunately, 🤗 Datasets is designed to be interoperable with libraries such as Pandas, NumPy, PyTorch, TensorFlow, and JAX. Let's take a look at how this works.

### From `Datasets` to `DataFrames` and back

In [41]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/tfcY1067A5Q" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


To enable the conversion between various third-party libraries, 🤗 Datasets provides a `Dataset.set_format()` function. This function only changes the *output format* of the dataset, so you can easily switch to another format without affecting the underlying *data format*, which is Apache Arrow. The formatting is done in place. To demonstrate, let's convert our dataset to Pandas:

In [42]:
drug_dataset.set_format("pandas")
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

Now when we access elements of the dataset we get a `pandas.DataFrame` instead of a dictionary:

In [43]:
drug_dataset["train"][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


Let's create a `pandas.DataFrame` for the whole training set by selecting all the elements of `drug_dataset["train"]`:

In [44]:
train_df = drug_dataset["train"][:]
train_df

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89
3,35696,Buprenorphine / naloxone,opiate dependence,"""Suboxone has completely turned my life around...",9.0,"November 27, 2016",37,124
4,155963,Cialis,benign prostatic hyperplasia,"""2nd day on 5mg started to work with rock hard...",2.0,"November 28, 2015",43,68
...,...,...,...,...,...,...,...,...
138509,164345,Junel 1.5 / 30,birth control,"""This would be my second month on Junel. I've ...",6.0,"May 27, 2015",0,71
138510,191035,Campral,alcohol dependence,"""I wrote my first report in Mid-October of 201...",10.0,"May 31, 2015",125,127
138511,127085,Metoclopramide,nausea/vomiting,"""I was given this in IV before surgey. I immed...",1.0,"November 1, 2011",34,50
138512,47128,Thyroid desiccated,underactive thyroid,"""I've been on thyroid medication 49 years, I s...",10.0,"September 19, 2015",79,136


> <font color="darkgreen">🚨 Under the hood, `Dataset.set_format()` changes the return format for the dataset's `__getitem__()` dunder method. This means that when we want to create a new object like `train_df` from a `Dataset` in the `"pandas"` format, we need to slice the whole dataset to obtain a `pandas.DataFrame`. You can verify for yourself that the type of `drug_dataset["train"]` is `Dataset`, irrespective of the output format.</font>

From here we can use all the Pandas functionality that we want. For example, we can do fancy chaining to compute the class distribution among the `condition` entries:

In [45]:
frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "condition": "frequency"})
)
frequencies.head()

,frequency,count
0,birth control,27655
1,depression,8023
2,acne,5209
3,anxiety,4991
4,pain,4744


And once we're done with our Pandas analysis, we can always create a new `Dataset` object by using the `Dataset.from_pandas()` function as follows:

In [46]:
from datasets import Dataset
freq_dataset = Dataset.from_pandas(frequencies)
freq_dataset

Dataset({
    features: ['frequency', 'count'],
    num_rows: 819
})

> ✏️ Try it out! <font color="darkgreen">Compute the average rating per drug and store the result in a new `Dataset`.</font>

In [47]:
# Trying it out
## https://stackoverflow.com/questions/30482071/how-to-calculate-mean-values-grouped-on-another-column-in-pandas
## https://stackoverflow.com/questions/17141558/how-to-sort-a-dataframe-in-python-pandas-by-two-or-more-columns
ratings = (
    train_df
    .groupby("drugName", as_index=False)["rating"]
    .mean()
    .sort_values(["rating", "drugName"], ascending=[False, True])
)
ratings

,drugName,rating
0,A + D Cracked Skin Relief,10.0
1,A / B Otic,10.0
8,Abiraterone,10.0
12,Absorbine Jr.,10.0
17,Accolate,10.0
...,...,...
2999,Zileuton,1.0
3024,Zostavax,1.0
3025,Zoster vaccine live,1.0
3027,Zostrix Diabetic Foot Pain,1.0


This wraps up our tour of the various preprocessing techniques available in 🤗 Datasets. To round out the section, let's create a validation set to prepare the dataset for training a classifier on. Before doing so, we'll reset the output format of `drug_dataset` from `"pandas"` to `"arrow"`:

In [48]:
drug_dataset.reset_format()
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

### Creating a validation set

Although we have a test set we could use for evaluation, it's a good practice to leave the test set untouched and create a separate validation set during development. Once you are happy with the performance of your models on the validation set, you can do a final sanity check on the test set. This process helps mitigate the risk that you'll overfit to the test set and deploy a model that fails on real-world data.

🤗 Datasets provides a `Dataset.train_test_split()` function that is based on the famous functionality from `scikit-learn`. Let's use it to split our training set into `train` and `validation` splits (we set the seed argument for reproducibility):

In [49]:
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

Great, we've now prepared a dataset that's ready for training some models on! In section 5 we'll show you how to upload datasets to the Hugging Face Hub, but for now let's cap off our analysis by looking at a few ways you can save datasets on your local machine.

### Saving a dataset

In [50]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/blF9uxYcKHo" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Although 🤗 Datasets will cache every downloaded dataset and the operations performed on it, there are times when you'll want to save a dataset to disk (e.g., in case the cache gets deleted). As shown in the table below, 🤗 Datasets provides three main functions to save your dataset in different formats:

|Data format|Function|
|-----------|--------|
|Arrow|`Dataset.save_to_disk()`|
|CSV|`Dataset.to_csv()`|
|JSON|`Dataset.to_json()`|

For example, let's save our cleaned dataset in the Arrow format:

In [51]:
drug_dataset_clean.save_to_disk("sections/section_5/data/drug-reviews")

Saving the dataset (0/1 shards):   0%|          | 0/110811 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27703 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/46108 [00:00<?, ? examples/s]

This will create a directory with the following structure:
```
drug-reviews/
├── dataset_dict.json
├── test
│   ├── dataset.arrow
│   ├── dataset_info.json
│   └── state.json
├── train
│   ├── dataset.arrow
│   ├── dataset_info.json
│   ├── indices.arrow
│   └── state.json
└── validation
    ├── dataset.arrow
    ├── dataset_info.json
    ├── indices.arrow
    └── state.json
```
where we can see that each split is associated with its own *dataset.arrow* table, and some metadata in *dataset_info.json* and *state.json*. You can think of the Arrow format as a fancy table of columns and rows that is optimized for building high-performance applications that process and transport large datasets.

Once the dataset is saved, we can load it by using the `load_from_disk()` function as follows:

In [52]:
from datasets import load_from_disk
drug_dataset_reloaded = load_from_disk("sections/section_5/data/drug-reviews")
drug_dataset_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

For the CSV and JSON formats, we have to store each split as a separate file. One way to do this is by iterating over the keys and values in the `DatasetDict` object:

In [53]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format:   0%|          | 0/111 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/28 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/47 [00:00<?, ?ba/s]

This saves each split in [JSON Lines format](https://jsonlines.org/), where each row in the dataset is stored as a single line of JSON. Here's what the first example looks like:

In [54]:
!mv "drug-reviews-test.jsonl" "sections/section_5/data"
!mv "drug-reviews-train.jsonl" "sections/section_5/data"
!mv "drug-reviews-validation.jsonl" "sections/section_5/data"
!head -n 1 "sections/section_5/data/drug-reviews-train.jsonl"

{"patient_id":89879,"drugName":"Cyclosporine","condition":"keratoconjunctivitis sicca","review":"\"I have used Restasis for about a year now and have seen almost no progress.  For most of my life I've had red and bothersome eyes. After trying various eye drops, my doctor recommended Restasis.  He said it typically takes 3 to 6 months for it to really kick in but it never did kick in.  When I put the drops in it burns my eyes for the first 30 - 40 minutes.  I've talked with my doctor about this and he said it is normal but should go away after some time, but it hasn't. Every year around spring time my eyes get terrible irritated  and this year has been the same (maybe even worse than other years) even though I've been using Restasis for a year now. The only difference I notice was for the first couple weeks, but now I'm ready to move on.\"","rating":2.0,"date":"April 20, 2013","usefulCount":69,"review_length":147}


We can then use the techniques from [section 2](https://huggingface.co/course/chapter5/2) to load the JSON files as follows:

In [55]:
data_files = {
    "train": "sections/section_5/data/drug-reviews-train.jsonl",
    "validation": "sections/section_5/data/drug-reviews-validation.jsonl",
    "test": "sections/section_5/data/drug-reviews-test.jsonl",
}
drug_dataset_reloaded = load_dataset("json", data_files=data_files)
drug_dataset_reloaded

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

And that's it for our excursion into data wrangling with 🤗 Datasets! Now that we have a cleaned dataset for training a model on, here are a few ideas that you could try out:
1. Use the techniques from [Chapter 3](https://huggingface.co/course/chapter3) to train a classifier that can predict the patient condition based on the drug review.
1. Use the `summarization` pipeline from [Chapter 1](https://huggingface.co/course/chapter1) to generate summaries of the reviews.

Next, we'll take a look at how 🤗 Datasets can enable you to work with huge datasets without blowing up your laptop!

## [Big data? 🤗 Datasets to the rescue!](https://huggingface.co/course/chapter5/4?fw=pt)

Nowadays it is not uncommon to find yourself working with multi-gigabyte datasets, especially if you're planning to pretrain a transformer like BERT or GPT-2 from scratch. In these cases, even *loading* the data can be a challenge. For example, the WebText corpus used to pretrain GPT-2 consists of over 8 million documents and 40 GB of text — loading this into your laptop's RAM is likely to give it a heart attack!

Fortunately, 🤗 Datasets has been designed to overcome these limitations. It frees you from memory management problems by treating datasets as *memory-mapped* files, and from hard drive limits by *streaming* the entries in a corpus.

In [56]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/JwISwTCPPWo" allowfullscreen></iframe>')

In this section we'll explore these features of 🤗 Datasets with a huge 825 GB corpus known as [the Pile](https://pile.eleuther.ai/). Let's get started!

### What is the Pile?

The Pile is an English text corpus that was created by [EleutherAI](https://www.eleuther.ai/) for training large-scale language models. It includes a diverse range of datasets, spanning scientific articles, GitHub code repositories, and filtered web text. The training corpus is available in [14 GB chunks](https://mystic.the-eye.eu/public/AI/pile/), and you can also download several of the [individual components](https://mystic.the-eye.eu/public/AI/pile_preliminary_components/). Let's start by taking a look at the PubMed Abstracts dataset, which is a corpus of abstracts from 15 million biomedical publications on [PubMed](https://pubmed.ncbi.nlm.nih.gov/). The dataset is in [JSON Lines format](https://jsonlines.org/) and is compressed using the `zstandard` library, so first we need to install that:

In [57]:
# the following command needs to run only once
#!pip install zstandard

Next, we can load the dataset using the method for remote files that we learned in [section 2](https://huggingface.co/course/chapter5/2):

In [58]:
import os
from dotenv import load_dotenv
load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
from huggingface_hub import login
login(token=HUGGINGFACE_TOKEN)
import zstandard
from datasets import load_dataset
data_files = "https://huggingface.co/datasets/mdroth/huggingface-course_section-5_zst/resolve/main/data/PubMed-200k-RTC_train.jsonl.zst"
print(data_files)
pubmed_dataset = load_dataset("json", data_files=data_files, split="train")
pubmed_dataset

https://huggingface.co/datasets/mdroth/huggingface-course_section-5_zst/resolve/main/data/PubMed-200k-RTC_train.jsonl.zst


Dataset({
    features: ['abstract_text', 'target'],
    num_rows: 2211861
})

We can see that there are 2,211,861 rows and 2 columns in our dataset — that's a lot!
> <font color="darkgreen">✎ By default, 🤗 Datasets will decompress the files needed to load a dataset. If you want to preserve hard drive space, you can pass `DownloadConfig(delete_extracted=True)` to the `download_config` argument of `load_dataset()`. See the [documentation](https://huggingface.co/docs/datasets/package_reference/builder_classes.html?#datasets.utils.DownloadConfig) for more details.</font>

Let's inspect the contents of the first example:

In [59]:
pubmed_dataset[0]

{'abstract_text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
 'target': 'BACKGROUND'}

Okay, this looks like the abstract from a medical article. Now let's see how much RAM we've used to load the dataset!

### The magic of memory mapping
A simple way to measure memory usage in Python is with the [`psutil`](https://psutil.readthedocs.io/en/latest/) library, which can be installed with `pip` as follows:

In [60]:
# the following command needs to run only once
#!pip install psutil

It provides a `Process` class that allows us to check the memory usage of the current process as follows:

In [61]:
import psutil
# Process.memory_info is expressed in bytes, so convert to megabytes
print(f"RAM used: {psutil.Process().memory_info().rss / (1024**2):.2f} MB")

RAM used: 1203.63 MB


Here, the `rss` attribute refers to the *resident set size*, which is the fraction of memory that a process occupies in RAM. This measurement also includes the memory used by the Python interpreter and the libraries we've loaded, so the actual amount of memory used to load the dataset is a bit smaller. For comparison, let's see how large the dataset is on disk, using the `dataset_size` attribute. Since the result is expressed in bytes like before, we need to manually convert it to mega- or gigabytes:

In [62]:
print(f"Number of files in dataset : {pubmed_dataset.dataset_size}")
size_mb = pubmed_dataset.dataset_size / (1024**3)
print(f"Dataset size (cache file) : {size_mb:.2f} GB")

Number of files in dataset : 368440643
Dataset size (cache file) : 0.34 GB


Nice — despite it being almost 0.34 GB large, we're able to load and access the dataset with much less RAM!
> ✏️ Try it out! <font color="darkgreen">Pick one of the subsets from the Pile that is larger than your laptop or desktop's RAM, load it with 🤗 Datasets, and measure the amount of RAM used. Note that to get an accurate measurement, you'll want to do this in a new process. You can find the decompressed sizes of each subset in Table 1 of the Pile paper.</font>

In [63]:
# Trying it out
data_files = "https://huggingface.co/datasets/mdroth/huggingface-course_section-5_zst/resolve/main/data/LegalText-classification_train.jsonl.zst"
FreeLaw_dataset = load_dataset("json", data_files=data_files, split="train")
print(f"Number of files in dataset : {FreeLaw_dataset.dataset_size}")
size_gb = FreeLaw_dataset.dataset_size / (1024**3)
print(f"Dataset size (cache file) : {size_gb:.2f} GB")

Number of files in dataset : 68062254
Dataset size (cache file) : 0.06 GB


If you're familiar with Pandas, this result might come as a surprise because of Wes Kinney's famous [rule of thumb](https://wesmckinney.com/blog/apache-arrow-pandas-internals/) that you typically need 5 to 10 times as much RAM as the size of your dataset. So how does 🤗 Datasets solve this memory management problem? 🤗 Datasets treats each dataset as a [memory-mapped file](https://en.wikipedia.org/wiki/Memory-mapped_file), which provides a mapping between RAM and filesystem storage that allows the library to access and operate on elements of the dataset without needing to fully load it into memory.

Memory-mapped files can also be shared across multiple processes, which enables methods like `Dataset.map()` to be parallelized without needing to move or copy the dataset. Under the hood, these capabilities are all realized by the [Apache Arrow](https://arrow.apache.org/) memory format and [`pyarrow`](https://arrow.apache.org/docs/python/index.html) library, which make the data loading and processing lightning fast. (For more details about Apache Arrow and comparisons to Pandas, check out [Dejan Simic's blog post](https://towardsdatascience.com/apache-arrow-read-dataframe-with-zero-memory-69634092b1a).) To see this in action, let's run a little speed test by iterating over all the elements in the PubMed Abstracts dataset:

In [64]:
import timeit
code_snippet = """
batch_size = 1000
for idx in range(0, len(pubmed_dataset), batch_size):
    _ = pubmed_dataset[idx:idx + batch_size]
"""
time = timeit.timeit(stmt=code_snippet, number=1, globals=globals())
size_gb = round(size_gb, 2)
time = round(time, 2)
size_gb_per_s = round(size_gb/time, 4)
print(f"Iterated over {len(pubmed_dataset)} examples (about {size_gb} GB) in {time}s, i.e., {size_gb_per_s} GB/s")

Iterated over 2211861 examples (about 0.06 GB) in 3.5s, i.e., 0.0171 GB/s


Here we've used Python's `timeit` module to measure the execution time taken by `code_snippet`. You'll typically be able to iterate over a dataset at speeds of a few tenths of a GB/s to several GB/s. This works great for the vast majority of applications, but sometimes you'll have to work with a dataset that is too large to even store on your laptop's hard drive. For example, if we tried to download the Pile in its entirety, we'd need 825 GB of free disk space! To handle these cases, 🤗 Datasets provides a streaming feature that allows us to download and access elements on the fly, without needing to download the whole dataset. Let's take a look at how this works.
> <font color="darkgreen">💡 In Jupyter notebooks, you can also time cells using the [`%%timeit` magic function](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit).</font>

### Streaming datasets
To enable dataset streaming you just need to pass the `streaming=True` argument to the `load_dataset()` function. For example, let's load the PubMed Abstracts dataset again, but in streaming mode:

In [65]:
data_files = "https://huggingface.co/datasets/mdroth/huggingface-course_section-5_zst/resolve/main/data/PubMed-200k-RTC_train.jsonl.zst"
pubmed_dataset_streamed = load_dataset("json", data_files=data_files, split="train", streaming=True)
pubmed_dataset_streamed

IterableDataset({
    features: ['abstract_text', 'target'],
    num_shards: 1
})

Instead of the familiar `Dataset` that we've encountered elsewhere in this chapter, the object returned with `streaming=True` is an `IterableDataset`. As the name suggests, to access the elements of an `IterableDataset` we need to iterate over it. We can access the first element of our streamed dataset as follows:

In [66]:
next(iter(pubmed_dataset_streamed))

{'abstract_text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
 'target': 'BACKGROUND'}

The elements from a streamed dataset can be processed on the fly using `IterableDataset.map()`, which is useful during training if you need to tokenize the inputs. The process is exactly the same as the one we used to tokenize our dataset in [Chapter 3](https://huggingface.co/course/chapter3), with the only difference being that outputs are returned one by one:

In [67]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenized_dataset = pubmed_dataset_streamed.map(lambda x: tokenizer(x["abstract_text"]))
next(iter(tokenized_dataset))

{'abstract_text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
 'target': 'BACKGROUND',
 'input_ids': [101,
  1996,
  14053,
  1997,
  9820,
  2004,
  1037,
  11888,
  4650,
  2965,
  2008,
  2111,
  2542,
  2007,
  9820,
  2024,
  3223,
  2000,
  2202,
  2062,
  5368,
  2005,
  1996,
  2969,
  1011,
  2968,
  1997,
  2037,
  4650,
  1010,
  2164,
  2437,
  3558,
  1010,
  6832,
  1998,
  2591,
  24081,
  1012,
  102],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

> <font color="darkgreen">💡 To speed up tokenization with streaming you can pass `batched=True`, as we saw in the last section. It will process the examples batch by batch; the default batch size is 1,000 and can be specified with the `batch_size` argument.</font>

You can also shuffle a streamed dataset using `IterableDataset.shuffle()`, but unlike `Dataset.shuffle()` this only shuffles the elements in a predefined `buffer_size`:

In [68]:
shuffled_dataset = pubmed_dataset_streamed.shuffle(buffer_size=10_000, seed=42)
next(iter(shuffled_dataset))

{'abstract_text': 'Safety and reactogenicity of a new heptavalent DTPw-HBV/Hib-MenAC ( diphtheria , tetanus , whole cell pertussis-hepatitis B virus/Haemophilus influenzae type b-Neisseria meningitidis serogroups A and C ) vaccine was compared with a widely used pentavalent DTPw-HBV/Hib vaccine .',
 'target': 'OBJECTIVE'}

In this example, we selected a random example from the first 10,000 examples in the buffer. Once an example is accessed, its spot in the buffer is filled with the next example in the corpus (i.e., the 10,001st example in the case above). You can also select elements from a streamed dataset using the `IterableDataset.take()` and `IterableDataset.skip()` functions, which act in a similar way to `Dataset.select()`. For example, to select the first 5 examples in the PubMed Abstracts dataset we can do the following:

In [69]:
dataset_head = pubmed_dataset_streamed.take(5)
list(dataset_head)

[{'abstract_text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
  'target': 'BACKGROUND'},
 {'abstract_text': 'This paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .',
  'target': 'BACKGROUND'},
 {'abstract_text': 'This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .',
  'target': 'METHODS'},
 {'abstract_text': "The intervention group will participate in the online group program ` Positive Outlook ' .",
  'target': 'METHODS'},
 {'abstract_text': 'The program is based on self-efficacy theory and uses a self-management approach to enhance skills , confidence and abilities to manag

Similarly, you can use the `IterableDataset.skip()` function to create training and validation splits from a shuffled dataset as follows:

In [70]:
# Skip the first 1,000 examples and include the rest in the training set
train_dataset = shuffled_dataset.skip(1000)
# Take the first 1,000 examples for the validation set
validation_dataset = shuffled_dataset.take(1000)
validation_dataset

IterableDataset({
    features: ['abstract_text', 'target'],
    num_shards: 1
})

Let's round out our exploration of dataset streaming with a common application: combining multiple datasets together to create a single corpus. 🤗 Datasets provides an `interleave_datasets()` function that converts a list of `IterableDataset` objects into a single `IterableDataset`, where the elements of the new dataset are obtained by alternating among the source examples. This function is especially useful when you're trying to combine large datasets, so as an example let's stream the FreeLaw subset of the Pile, which is a 51 GB dataset of legal opinions from US courts:

In [71]:
law_dataset_streamed = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/mdroth/huggingface-course_section-5_zst/resolve/main/data/LegalText-classification_train.jsonl.zst",
    split="train",
    streaming=True,
)
next(iter(law_dataset_streamed))

{'case_outcome': 'cited',
 'case_title': 'Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Ltd (No 2) [2002] FCA 224 ; (2002) 190 ALR 121',
 'case_text': 'Ordinarily that discretion will be exercised so that costs follow the event and are awarded on a party and party basis. A departure from normal practice to award indemnity costs requires some special or unusual feature in the case: Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Ltd (No 2) [2002] FCA 224 ; (2002) 190 ALR 121 at [11] (Weinberg J) citing Colgate Palmolive Co v Cussons Pty Ltd (1993) 46 FCR 225 at 233 (Sheppard J).'}

This dataset is large enough to stress the RAM of most laptops, yet we've been able to load and access it without breaking a sweat! Let's now combine the examples from the FreeLaw and PubMed Abstracts datasets with the `interleave_datasets()` function:

In [72]:
from itertools import islice
from datasets import interleave_datasets
combined_dataset = interleave_datasets([pubmed_dataset_streamed, law_dataset_streamed])
list(islice(combined_dataset, 2))

[{'abstract_text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
  'target': 'BACKGROUND',
  'case_outcome': None,
  'case_title': None,
  'case_text': None},
 {'abstract_text': None,
  'target': None,
  'case_outcome': 'cited',
  'case_title': 'Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Ltd (No 2) [2002] FCA 224 ; (2002) 190 ALR 121',
  'case_text': 'Ordinarily that discretion will be exercised so that costs follow the event and are awarded on a party and party basis. A departure from normal practice to award indemnity costs requires some special or unusual feature in the case: Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Ltd (No 2) [2002] FCA 224 ; (2002) 190 ALR 121 at [11] (Weinberg J) citing Colgate Palmolive Co v Cussons Pty Ltd (1993) 46 FCR 225 at 233 (Sheppard J).'}]

Here we've used the `islice()` function from Python's `itertools` module to select the first two examples from the combined dataset, and we can see that they match the first examples from each of the two source datasets.

Finally, if you want to stream the [Pile (deduplicated)](https://huggingface.co/datasets/gmongaras/EleutherAI_the_pile_deduplicated)  in its 825 GB entirety, you can grab all the prepared files as follows:

In [73]:
from datasets import load_dataset
# note the "train-*-of00083.parquet" in the next line
data_files = "https://huggingface.co/datasets/gmongaras/EleutherAI_the_pile_deduplicated/resolve/main/data/train-*-of-00083.parquet"
# load the dataset in streaming mode using the built-in parquet reader
streaming_dataset = load_dataset("parquet", data_files=data_files, streaming=True)
print(streaming_dataset["train"])
# custom addition for iterating over the streaming dataset and printing the first 4 examples
for i, instance in enumerate(streaming_dataset["train"]):
    print("#"*14)
    print(f"# instance {i} #")
    print("#"*14)
    print(f"\n{instance["text"][:1000]} ...\n")
    if i == 3: # stop after printing 4 examples
        break

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

IterableDataset({
    features: ['text'],
    num_shards: 83
})
##############
# instance 0 #
##############

It is done, and submitted. You can play “Survival of the Tastiest” on Android, and on the web. Playing on the web works, but you have to simulate multi-touch for table moving and that can be a bit confusing.

There’s a lot I’d like to talk about. I’ll go through every topic, insted of making the typical what went right/wrong list.

Concept

Working over the theme was probably one of the hardest tasks I had to face.

Originally, I had an idea of what kind of game I wanted to develop, gameplay wise – something with lots of enemies/actors, simple graphics, maybe set in space, controlled from a top-down view. I was confident I could fit any theme around it.

In the end, the problem with a theme like “Evolution” in a game is that evolution is unassisted. It happens through several seemingly random mutations over time, with the most apt permutation surviving. This genetic car simulat

> ✏️ Try it out! <font color="darkgreen">Use one of the large Common Crawl corpora like [`mc4`](https://huggingface.co/datasets/mc4) or [`oscar`](https://huggingface.co/datasets/oscar) to create a streaming multilingual dataset that represents the spoken proportions of languages in a country of your choice. For example, the four national languages in Switzerland are German, French, Italian, and Romansh, so you could try creating a Swiss corpus by sampling the Oscar subsets according to their spoken proportion.</font>

In [74]:
# Trying it out
# The following languages are supported: bg, cs, da, de, el, en, es, et, fi, fr, ga, hu, it, lt, lv, mt, nl, pl, pt, ro, sk, sl, sv
# According to ChatGPT, 96.5% of people in Germany speak German and 60.5% speak English
# 96.5 / 60.5 = 1.5950 = 15950 / 10000
mc4_legal_dataset_de = load_dataset("joelito/mc4_legal", "de", split="train", streaming=True).remove_columns(["url", "timestamp", "matches"])
mc4_legal_dataset_en = load_dataset("joelito/mc4_legal", "en", split="train", streaming=True).remove_columns(["url", "timestamp", "matches"])
mc4_legal_dataset_de, mc4_legal_dataset_en

Loading Dataset Infos from /home/matthias/.cache/huggingface/modules/datasets_modules/datasets/joelito--mc4_legal/455b0c3963b4c25452e226c39b79bcbb7f8c2fc30ca86b7a930b157a0618814b
Loading Dataset Infos from /home/matthias/.cache/huggingface/modules/datasets_modules/datasets/joelito--mc4_legal/455b0c3963b4c25452e226c39b79bcbb7f8c2fc30ca86b7a930b157a0618814b


(IterableDataset({
     features: ['index', 'text'],
     num_shards: 2
 }),
 IterableDataset({
     features: ['index', 'text'],
     num_shards: 2
 }))

These are the chosen datasets. The following function takes two lists as arguments: The first list is a list of iterable datasets and the second is an equally long list of instances. The latter list shall be used to control the number of instances stemming from the passed datasets.

In [75]:
# Still trying it out
import random
from itertools import islice
def interleaved_shuffle(datasets, samples_per_dataset, buffer_size=1000):
    """
    Streams from multiple datasets and interleaves their samples with partial shuffling.
    Arguments:
    - datasets: list of streaming datasets (or iterators)
    - samples_per_dataset: list of integers (how many items to take from each)
    - buffer_size: number of items to buffer before shuffling
    Returns:
    - a generator that yields interleaved and partially shuffled examples
    """
    # Make sure we're getting actual samples from each dataset, not the iterators themselves!
    iterators = [islice(ds, n) for ds, n in zip(datasets, samples_per_dataset)]
    # This will store our current buffer of samples
    buffer = []
    # We'll loop until all iterators are exhausted
    # Instead of zip(), loop over each iterator independently in round-robin style
    while iterators:
        for i, it in enumerate(iterators[:]):  # iterate over a *copy* of the list
            try:
                item = next(it)  # try to get next item
                buffer.append(item)
            except StopIteration:
                # Remove exhausted iterator
                iterators.remove(it)
            # If buffer is full, shuffle and yield
            if len(buffer) >= buffer_size:
                random.shuffle(buffer)
                for sample in buffer:
                    yield sample
                buffer.clear()
    # Final flush of any remaining items
    if buffer:
        random.shuffle(buffer)
        for sample in buffer:
            yield sample
from itertools import islice

# Create the interleaved and shuffled streaming dataset
mc4_legal_de_en_interleaved_shuffled_streaming_dataset = interleaved_shuffle(
    [mc4_legal_dataset_de, mc4_legal_dataset_en],
    [15950, 10000]
)
# Now print the actual first m examples and each text only until the n-th character
m = 3
n = 1000
for i, example in enumerate(islice(mc4_legal_de_en_interleaved_shuffled_streaming_dataset, m), 1):
    print(f"\n{i}.\n{example['text'][:n]} ...")


1.
Europäischer Gerichtshof, Urteil vom 08.05.2003 mit dem Az.: C-14/02	/* Banner Ads */
Aktenzeichen: C-14/02
Rechtsgebiete: EGV, Richtlinie 73/23/EWG, Richtlinie 89/336/EWG, Richtlinie 1999/5/EWG
Richtlinie 1999/5/EWG
1. Aus dem Wortlaut und der Zielsetzung der Richtlinie 73/23 zur Angleichung der Rechtsvorschriften der Mitgliedstaaten betreffend elektrische Betriebsmittel zur Verwendung innerhalb bestimmter Spannungsgrenzen, der Richtlinie 89/336 zur Angleichung der Rechtsvorschriften der Mitgliedstaaten über die elektromagnetische Verträglichkeit sowie der Richtlinie 1999/5 über Funkanlagen und Telekommunikationseinrichtungen und die gegenseitige Anerkennung ihrer Konformität ergibt sich, dass sie in ihrem jeweiligen Anwendungsbereich eine vollständige Harmonisierung bezwecken. Folglich müssen die Mitgliedstaaten diesen Richtlinien in den diesen unterliegenden Bereichen voll nachkommen und dürfen zuwiderlaufende nationale Bestimmungen nicht beibehalten.
Artikel 3 der Richtlinie 73

You now have all the tools you need to load and process datasets of all shapes and sizes — but unless you're exceptionally lucky, there will come a point in your NLP journey where you'll have to actually create a dataset to solve the problem at hand. That's the topic of the next section!

## [Creating your own dataset](https://huggingface.co/course/chapter5/5?fw=pt)

Sometimes the dataset that you need to build an NLP application doesn't exist, so you'll need to create it yourself. In this section we'll show you how to create a corpus of [GitHub issues](https://github.com/features/issues/), which are commonly used to track bugs or features in GitHub repositories. This corpus could be used for various purposes, including:
- Exploring how long it takes to close open issues or pull requests
- Training a *multilabel classifier* that can tag issues with metadata based on the issue's description (e.g., "bug", "enhancement", or "question")
- Creating a semantic search engine to find which issues match a user's query

Here we'll focus on creating the corpus, and in the next section we'll tackle the semantic search application. To keep things meta, we'll use the GitHub issues associated with a popular open source project: 🤗 Datasets! Let's take a look at how to get the data and explore the information contained in these issues.

### Getting the data

You can find all the issues in 🤗 Datasets by navigating to the repository's [Issues tab](https://github.com/huggingface/datasets/issues). As shown in the following screenshot, at the time of writing there were 331 open issues and 668 closed ones.

<img style="float=center;" src="sections/section_5/images/huggingfaceDatasets.png" width="80%">

If you click on one of these issues you'll find it contains a title, a description, and a set of labels that characterize the issue. An example is shown in the screenshot below.

<img style="float=center;" src="sections/section_5/images/enableStreaming.png" width="90%">

To download all the repository's issues, we'll use the [GitHub REST API](https://docs.github.com/en/rest) to poll the [`Issues` endpoint](https://docs.github.com/en/rest/reference/issues#list-repository-issues). This endpoint returns a list of JSON objects, with each object containing a large number of fields that include the title and description as well as metadata about the status of the issue and so on.

A convenient way to download the issues is via the `requests` library, which is the standard way for making HTTP requests in Python. You can install the library by running:

In [76]:
# the following command needs to run only once 
#!pip install requests

Once the library is installed, you can make GET requests to the `Issues` endpoint by invoking the `requests.get()` function. For example, you can run the following command to retrieve the first issue on the first page:

In [77]:
import requests
url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

The `response` object contains a lot of useful information about the request, including the HTTP status code:

In [78]:
response.status_code

200

where a `200` status means the request was successful (you can find a list of possible HTTP status codes [here](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes)). What we are really interested in, though, is the *payload*, which can be accessed in various formats like bytes, strings, or JSON. Since we know our issues are in JSON format, let's inspect the payload as follows:

In [79]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/7472',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/7472/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/7472/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/7472/events',
  'html_url': 'https://github.com/huggingface/datasets/issues/7472',
  'id': 2937607272,
  'node_id': 'I_kwDODunzps6vGFRo',
  'number': 7472,
  'title': 'Label casting during `map` process is canceled after the `map` process',
  'user': {'login': 'yoshitomo-matsubara',
   'id': 11156001,
   'node_id': 'MDQ6VXNlcjExMTU2MDAx',
   'avatar_url': 'https://avatars.githubusercontent.com/u/11156001?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/yoshitomo-matsubara',
   'html_url': 'https://github.com/yoshitomo-matsubara',
   'followers_url': 'https://api.gith

Whoa, that's a lot of information! We can see useful fields like `title`, `body`, and `number` that describe the issue, as well as information about the GitHub user who opened the issue.
> ✏️ Try it out! <font color="darkgreen">Click on a few of the URLs in the JSON payload above to get a feel for what type of information each GitHub issue is linked to.</font>

In [80]:
# Trying it out
try_str = """
The url https://github.com/huggingface/datasets/issues/7399 is about a potential bug in the IterableDatasetDict map function:
It might be missing the desc parameter.
Under `user` > `avatar_url`, the url https://avatars.githubusercontent.com/u/7976840?v=4 leads to a cute avatar image.
"""
print(try_str)


The url https://github.com/huggingface/datasets/issues/7399 is about a potential bug in the IterableDatasetDict map function:
It might be missing the desc parameter.
Under `user` > `avatar_url`, the url https://avatars.githubusercontent.com/u/7976840?v=4 leads to a cute avatar image.



As described in the GitHub [documentation](https://docs.github.com/en/rest/overview/resources-in-the-rest-api#rate-limiting), unauthenticated requests are limited to 60 requests per hour. Although you can increase the `per_page` query parameter to reduce the number of requests you make, you will still hit the rate limit on any repository that has more than a few thousand issues. So instead, you should follow GitHub's [instructions](https://docs.github.com/en/github/authenticating-to-github/creating-a-personal-access-token) on creating a *personal access token* so that you can boost the rate limit to 5,000 requests per hour. Once you have your token, you can include it as part of the request header:

In [81]:
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

> <font color="darkred">⚠️ Do not share a notebook with your `GITHUB_TOKEN` pasted in it. We recommend you delete the last cell once you have executed it to avoid leaking this information accidentally. Even better, store the token in an *.env* file and use the [`python-dotenv` library](https://github.com/theskumar/python-dotenv) to load it automatically for you as an environment variable</font>.

Now that we have our access token, let's create a function that can download all the issues from a GitHub repository:

In [113]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=1_000,
    rate_limit=100,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # Query with state=all to get both open and closed issues
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl")

Now when we call `fetch_issues()` it will download all the issues in batches to avoid exceeding GitHub's limit on the number of requests per hour; the result will be stored locally in a *repository_name-issues.jsonl* file, where each line is a JSON object that represents an issue. Let's use this function to grab all the issues from 🤗 Datasets:

In [ ]:
# Depending on your internet connection, this can take several minutes to run...
# The following line needs to run only once (run 'fetch_issues(repo="transformers")' for "Try it out" further below)
fetch_issues()                      # for the default datasets repo

  0%|          | 0/10 [00:00<?, ?it/s]

Reached GitHub rate limit. Sleeping for one hour ...


Once the issues are downloaded we can load them locally using our newfound skills from [section 2](https://huggingface.co/course/chaper5/2):

In [115]:
issues_dataset = load_dataset("json", data_files="sections/section_5/data/datasets-issues.jsonl", split="train")
issues_dataset

Using custom data configuration default-ad8f542b20e92ae9
Loading Dataset Infos from /home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/datasets/packaged_modules/json
Overwrite dataset info from restored data version if exists.
Loading Dataset info from /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092
Found cached dataset json (/home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092)
Loading Dataset info from /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'type', 'sub_issues_summary', 'active_lock_reason', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'draft', 'pull_request'],
    num_rows: 3000
})

Great, we've created our first dataset from scratch! But why are there several thousand issues when the [Issues tab](https://github.com/huggingface/datasets/issues) of the 🤗 Datasets repository only shows around 1,000 issues in total 🤔? As described in the GitHub [documentation](https://docs.github.com/en/rest/reference/issues#list-issues-assigned-to-the-authenticated-user), that's because we've downloaded all the pull requests as well:

> <i>"GitHub's REST API v3 considers every pull request an issue, but not every *issue* is a pull request. For this reason, "*Issues*" endpoints may return both issues and pull requests in the response. You can identify pull requests by the `pull_request` key. Be aware that the `id` of a pull request returned from "*Issues*" endpoints will be an issue id."</i>

Since the contents of issues and pull requests are quite different, let's do some minor preprocessing to enable us to distinguish between them.

### Cleaning up the data

The above snippet from GitHub's documentation tells us that the `pull_request` column can be used to differentiate between issues and pull requests. Let's look at a random sample to see what the difference is. As we did in [section 3](https://huggingface.co/course/chapter5/3), we'll chain `Dataset.shuffle()` and `Dataset.select()` to create a random sample and then zip the `html_url` and `pull_request` columns so we can compare the various URLs:

In [116]:
sample = issues_dataset.shuffle(seed=666).select(range(3))
# print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

Loading cached shuffled indices for dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-1246b532aba00c4b.arrow


>> URL: https://github.com/huggingface/datasets/issues/7323
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/pull/4630
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/4630', 'html_url': 'https://github.com/huggingface/datasets/pull/4630', 'diff_url': 'https://github.com/huggingface/datasets/pull/4630.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/4630.patch', 'merged_at': datetime.datetime(2022, 7, 5, 15, 8, 21)}

>> URL: https://github.com/huggingface/datasets/pull/4645
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/4645', 'html_url': 'https://github.com/huggingface/datasets/pull/4645', 'diff_url': 'https://github.com/huggingface/datasets/pull/4645.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/4645.patch', 'merged_at': datetime.datetime(2022, 7, 6, 15, 45, 5)}



Here we can see that each pull request is associated with various URLs, while ordinary issues have a `None` entry. We can use this distinction to create a new `is_pull_request` column that checks whether the `pull_request` field is `None` or not:

In [117]:
issues_dataset = issues_dataset.map(lambda x: {"is_pull_request": False if x["pull_request"] is None else True})
issues_dataset

Loading cached processed dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-329a0347ca0f3357.arrow


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'type', 'sub_issues_summary', 'active_lock_reason', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'draft', 'pull_request', 'is_pull_request'],
    num_rows: 3000
})

> ✏️ Try it out! <font color="darkgreen">Calculate the average time it takes to close issues in 🤗 Datasets. You may find the `Dataset.filter()` function useful to filter out the pull requests and open issues, and you can use the `Dataset.set_format()` function to convert the dataset to a `DataFrame` so you can easily manipulate the `created_at` and `closed_at` timestamps. For bonus points, calculate the average time it takes to close pull requests.</font>

In [118]:
# Trying it out
import datetime
## get closed issues and pull requests
print(f"total items:\t\t{issues_dataset.num_rows}")
closed_dataset = issues_dataset.filter(lambda x: x["closed_at"] is not None)
print(f"closed items:\t\t{closed_dataset.num_rows}")
closed_issues_dataset = closed_dataset.filter(lambda x: x["pull_request"] is None)
print(f"closed issues:\t\t{closed_issues_dataset.num_rows}")
closed_pullRequests_dataset = closed_dataset.filter(lambda x: x["pull_request"] is not None)
print(f"closed pull requests:\t{closed_pullRequests_dataset.num_rows}")
## define helper functions
### format time
def formatTime(tstr, value, unit, trail):
    if tstr!="" or value!=0:
        tstr += f"{int(value)}{unit}{trail}"
    return tstr
### turn seconds into time string
def secs2DHMS(secs):
    tstr=""
    days = secs // (24 * 3600)
    secs %= 24 * 3600
    tstr = formatTime(tstr, days, "d", " ")
    hours = secs // 3600
    secs %= 3600
    tstr = formatTime(tstr, hours, "h", ":")
    mins = secs // 60
    secs %= 60
    tstr = formatTime(tstr, mins, "m", ":")
    tstr = formatTime(tstr, secs, "s", "")
    return tstr
### get mean closing time (in seconds) of github issues or pull requests
def get_mean_closing_time(github_dataset):
    durations = []
    for item in github_dataset:
        start = str(item["created_at"])
        end = str(item["closed_at"])
        # https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes
        start_seconds = time.mktime(datetime.datetime.strptime(start, "%Y-%m-%d %H:%M:%S").timetuple())
        end_seconds = time.mktime(datetime.datetime.strptime(end, "%Y-%m-%d %H:%M:%S").timetuple())
        durations.append(end_seconds - start_seconds)
    mean = sum(durations) / len(durations)
    return secs2DHMS(mean)
## produce output
print(f"average duration until closing for GitHub issues:\t\t{get_mean_closing_time(closed_issues_dataset)}")
print(f"average duration until closing for GitHub pull requests:\t{get_mean_closing_time(closed_pullRequests_dataset)}")

Loading cached processed dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-be874956fbd0d54a.arrow
Loading cached processed dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-f043ed385cafdf91.arrow
Loading cached processed dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-3f61622913b6fa1b.arrow


total items:		3000
closed items:		2335
closed issues:		984
closed pull requests:	1351
average duration until closing for GitHub issues:		33d 4h:21m:24s
average duration until closing for GitHub pull requests:	12d 1h:53m:39s


Although we could proceed to further clean up the dataset by dropping or renaming some columns, it is generally a good practice to keep the dataset as "raw" as possible at this stage so that it can be easily used in multiple applications.

Before we push our dataset to the Hugging Face Hub, let's deal with one thing that's missing from it: the comments associated with each issue and pull request. We'll add them next with — you guessed it — the GitHub REST API!

### Augmenting the dataset

As shown in the following screenshot, the comments associated with an issue or pull request provide a rich source of information, especially if we're interested in building a search engine to answer user queries about the library.

<img style="float=center;" width="85%" src="sections/section_5/images/adding_a_dataset.png">

The GitHub REST API provides a [`Comments` endpoint](https://docs.github.com/en/rest/reference/issues#list-issue-comments) that returns all the comments associated with an issue number. Let's test the endpoint to see what it returns:

In [119]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

We can see that the comment is stored in the `body` field, so let's write a simple function that returns all the comments associated with an issue by picking out the `body` contents for each element in `response.json()`:

In [120]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]

# Test our function works as expected
get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

This looks good, so let's use `Dataset.map()` to add a new `comments` column to each issue in our dataset:

In [121]:
%%time
# Depending on your internet connection, this can take quite some time.
# And due to the GitHub API's rate limit, it might take several minutes to just see any progress, at all!
issues_with_comments_dataset = issues_dataset.map(lambda x: {"comments": get_comments(x["number"])})
issues_with_comments_dataset

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Caching processed dataset at /home/matthias/.cache/huggingface/datasets/json/default-ad8f542b20e92ae9/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-8159dbb3d5aba2e0.arrow


ConnectTimeout: HTTPSConnectionPool(host='api.github.com', port=443): Max retries exceeded with url: /repos/huggingface/datasets/issues/7461/comments (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7a264144f590>, 'Connection to api.github.com timed out. (connect timeout=None)'))

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1622320308', 'html_url': 'https://github.com/huggingface/datasets/issues/6008#issuecomment-1622320308', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/6008', 'id': 1622320308, 'node_id': 'IC_kwDODunzps5gsqS0', 'user': {'login': 'mariosasko', 'id': 47462742, 'node_id': 'MDQ6VXNlcjQ3NDYyNzQy', 'avatar_url': 'https://avatars.githubusercontent.com/u/47462742?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/mariosasko', 'html_url': 'https://github.com/mariosasko', 'followers_url': 'https://api.github.com/users/mariosasko/followers', 'following_url': 'https://api.github.com/users/mariosasko/following{/other_user}', 'gists_url': 'https://api.github.com/users/mariosasko/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/mariosasko/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/mariosasko/subscriptions', 'organizations_url': 'https://api.githu

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1621109784', 'html_url': 'https://github.com/huggingface/datasets/issues/6006#issuecomment-1621109784', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/6006', 'id': 1621109784, 'node_id': 'IC_kwDODunzps5goCwY', 'user': {'login': 'xipq', 'id': 115634163, 'node_id': 'U_kgDOBuRv8w', 'avatar_url': 'https://avatars.githubusercontent.com/u/115634163?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/xipq', 'html_url': 'https://github.com/xipq', 'followers_url': 'https://api.github.com/users/xipq/followers', 'following_url': 'https://api.github.com/users/xipq/following{/other_user}', 'gists_url': 'https://api.github.com/users/xipq/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/xipq/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/xipq/subscriptions', 'organizations_url': 'https://api.github.com/users/xipq/orgs', 'repos_url': 'https://api.gith

...

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1540233167', 'html_url': 'https://github.com/huggingface/datasets/issues/5832#issuecomment-1540233167', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/5832', 'id': 1540233167, 'node_id': 'IC_kwDODunzps5bzhfP', 'user': {'login': 'varungupta31', 'id': 51288316, 'node_id': 'MDQ6VXNlcjUxMjg4MzE2', 'avatar_url': 'https://avatars.githubusercontent.com/u/51288316?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/varungupta31', 'html_url': 'https://github.com/varungupta31', 'followers_url': 'https://api.github.com/users/varungupta31/followers', 'following_url': 'https://api.github.com/users/varungupta31/following{/other_user}', 'gists_url': 'https://api.github.com/users/varungupta31/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/varungupta31/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/varungupta31/subscriptions', 'organizations_url': 'h

[]
[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1538721453', 'html_url': 'https://github.com/huggingface/datasets/issues/5829#issuecomment-1538721453', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/5829', 'id': 1538721453, 'node_id': 'IC_kwDODunzps5btwat', 'user': {'login': 'mariosasko', 'id': 47462742, 'node_id': 'MDQ6VXNlcjQ3NDYyNzQy', 'avatar_url': 'https://avatars.githubusercontent.com/u/47462742?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/mariosasko', 'html_url': 'https://github.com/mariosasko', 'followers_url': 'https://api.github.com/users/mariosasko/followers', 'following_url': 'https://api.github.com/users/mariosasko/following{/other_user}', 'gists_url': 'https://api.github.com/users/mariosasko/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/mariosasko/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/mariosasko/subscriptions', 'organizations_url': 'https://api.gi

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1542128789', 'html_url': 'https://github.com/huggingface/datasets/issues/5827#issuecomment-1542128789', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/5827', 'id': 1542128789, 'node_id': 'IC_kwDODunzps5b6wSV', 'user': {'login': 'lhoestq', 'id': 42851186, 'node_id': 'MDQ6VXNlcjQyODUxMTg2', 'avatar_url': 'https://avatars.githubusercontent.com/u/42851186?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/lhoestq', 'html_url': 'https://github.com/lhoestq', 'followers_url': 'https://api.github.com/users/lhoestq/followers', 'following_url': 'https://api.github.com/users/lhoestq/following{/other_user}', 'gists_url': 'https://api.github.com/users/lhoestq/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/lhoestq/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/lhoestq/subscriptions', 'organizations_url': 'https://api.github.com/users/lhoestq/orgs

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1536620258', 'html_url': 'https://github.com/huggingface/datasets/issues/5825#issuecomment-1536620258', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/5825', 'id': 1536620258, 'node_id': 'IC_kwDODunzps5blvbi', 'user': {'login': 'mariosasko', 'id': 47462742, 'node_id': 'MDQ6VXNlcjQ3NDYyNzQy', 'avatar_url': 'https://avatars.githubusercontent.com/u/47462742?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/mariosasko', 'html_url': 'https://github.com/mariosasko', 'followers_url': 'https://api.github.com/users/mariosasko/followers', 'following_url': 'https://api.github.com/users/mariosasko/following{/other_user}', 'gists_url': 'https://api.github.com/users/mariosasko/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/mariosasko/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/mariosasko/subscriptions', 'organizations_url': 'https://api.githu

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/1536206488', 'html_url': 'https://github.com/huggingface/datasets/issues/5823#issuecomment-1536206488', 'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/5823', 'id': 1536206488, 'node_id': 'IC_kwDODunzps5bkKaY', 'user': {'login': 'lhoestq', 'id': 42851186, 'node_id': 'MDQ6VXNlcjQyODUxMTg2', 'avatar_url': 'https://avatars.githubusercontent.com/u/42851186?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/lhoestq', 'html_url': 'https://github.com/lhoestq', 'followers_url': 'https://api.github.com/users/lhoestq/followers', 'following_url': 'https://api.github.com/users/lhoestq/following{/other_user}', 'gists_url': 'https://api.github.com/users/lhoestq/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/lhoestq/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/lhoestq/subscriptions', 'organizations_url': 'https://api.github.com/users/lhoestq/orgs

The final step is to push our dataset to the Hub. Let's take a look at how we can do that.

### Uploading the dataset to the Hugging Face Hub

In [122]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/HaN6qCr_Afc" allowfullscreen></iframe>')

Now that we have our augmented dataset, it's time to push it to the Hub so we can share it with the community! Uploading a dataset is very simple: just like models and tokenizers from 🤗 Transformers, we can use a `push_to_hub()` method to push a dataset. To do that we need an authentication token, which can be obtained by first logging into the Hugging Face Hub with the `notebook_login()` function. But for simplicity, we use the `login()` function instead, since it allows us to run the entire notebook without having to pass a token manually for authentication.

In [123]:
# We have already authenticated above in order to push to the hub.
# HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=HUGGINGFACE_TOKEN)

The `notebook_login` function would create a widget where you can enter your username and an API token that would be saved in *~/.huggingface/token*. If you're running the code in a terminal, you can log in via the CLI instead:
```
huggingface-cli login
```
Once we've done this, we can upload our dataset by running:

In [124]:
issues_with_comments_dataset.push_to_hub("github_issues") # = "mdroth/github_issues_10" on the huggingface hub

NameError: name 'issues_with_comments_dataset' is not defined

From here, anyone can download the dataset by simply providing `load_dataset()` with the repository ID as the path argument:

In [125]:
remote_dataset = load_dataset("mdroth/github_issues", split="train")
remote_dataset

DatasetNotFoundError: Dataset 'mdroth/github_issues' doesn't exist on the Hub or cannot be accessed.

Cool, we've pushed our dataset to the Hub and it's available for others to use! There's just one important thing left to do: adding a dataset card that explains how the corpus was created and provides other useful information for the community.

> 💡 <font color="darkgreen"> You can also upload a dataset to the Hugging Face Hub directly from the terminal by using huggingface-cli and a bit of Git magic. See the 🤗 Datasets guide for details on how to do this.</font>

> ✏️ Try it out! <font color="darkgreen">Use your Hugging Face Hub username and password to obtain a token and create an empty repository called `github_issues`. Remember to **never save your credentials** in Colab or any other repository, as this information can be exploited by bad actors.</font>

In [126]:
# Trying it out
## data splits: test (20%), validation (16%), training (64%)
## current state: training = 100% => split off 16% for validation
## https://discuss.huggingface.co/t/how-to-split-main-dataset-into-train-dev-test-as-datasetdict/1090/9
from datasets import DatasetDict
train_validTest = issues_with_comments_dataset_10.train_test_split(shuffle=True, seed=42, test_size=0.36)
valid_test = train_validTest["test"].train_test_split(shuffle=True, seed=42, test_size=5/9)
github_issues_dataset_10 = DatasetDict({
    "train": train_validTest["train"],
    "valid": valid_test["train"],
    "test": valid_test["test"]
})
print(github_issues_dataset_10)
## pushing to hub
## https://discuss.huggingface.co/t/save-datasetdict-to-huggingface-hub/12075/4
github_issues_dataset_10.push_to_hub(repo_id="github_issues_10")

NameError: name 'issues_with_comments_dataset_10' is not defined

Create a [git submodule](https://git-scm.com/book/en/v2/Git-Tools-Submodules#) in preparation of the next steps.

In [127]:
import pathlib
local_repo_path = "sections/section_5/data"
submodule_path = pathlib.Path(local_repo_path) / "github_issues_10"
if not submodule_path.exists():
    os.system(f"cd {local_repo_path} && git submodule add https://huggingface.co/datasets/mdroth/github_issues_10")
else:
    print(f"The git submodule already exists here: {submodule_path}")

The git submodule already exists here: sections/section_5/data/github_issues_10


Next, let's clone the repository from the Hub to our local machine and copy our dataset file into it. 🤗 Hub provides a handy `Repository` class that wraps many of the common Git commands, so to clone the remote repository we simply need to provide the URL and local path we wish to clone to:

In [ ]:
# run {!git lfs update --force} if necessary
repo_url = "https://huggingface.co/datasets/mdroth/github_issues_10/tree/main"
# https://huggingface.co/docs/huggingface_hub/main/en/package_reference/repository
repo_url = "https://huggingface.co/datasets/mdroth/github_issues_10"
#repo_url = "mdroth/github_issues_10"
from huggingface_hub import Repository
repo = Repository(local_dir="sections/section_5/data/github_issues_10", clone_from=repo_url)
repo.git_pull(lfs=True)
#!cp "sections/section_5/data/issues-datasets-with-comments.jsonl" "sections/section_5/data/github-issues/"
repo

By default, various file extensions (such as *.bin*, *.gz*, and *.zip*) are tracked with Git LFS so that large files can be versioned within the same Git workflow. You can find a list of tracked file extensions inside the repository's *.gitattributes* file. To include the JSON Lines format in the list, we can run the following command:

In [ ]:
repo.lfs_track("*.jsonl")

Then we can use `Repository.push_to_hub()` to push the dataset to the Hub:

In [ ]:
repo.push_to_hub()

If we navigate to the URL contained in `repo_url`, we should now see that our dataset file has been uploaded.

<img style="float=center;" width="70%" src="sections/section_5/images/lewtun_github_issues.png">

From here, anyone can download the dataset by simply providing `load_dataset()` with the repository ID as the `path` argument:

In [ ]:
remote_dataset = load_dataset("mdroth/github_issues_10")
remote_dataset

Cool, we've pushed our dataset to the Hub and it's available for others to use! There's just one important thing left to do: adding a *dataset card* that explains how the corpus was created and provides other useful information for the community.
> <font color="darkgreen">💡 You can also upload a dataset to the Hugging Face Hub directly from the terminal by using `huggingface-cli` and a bit of Git magic. See the [🤗 Datasets guide](https://huggingface.co/docs/datasets/share.html#add-a-community-dataset) for details on how to do this.</font>

### Creating a dataset card

Well-documented datasets are more likely to be useful to others (including your future self!), as they provide the context to enable users to decide whether the dataset is relevant to their task and to evaluate any potential biases in or risks associated with using the dataset.

On the Hugging Face Hub, this information is stored in each dataset repository's *README.md* file. There are two main steps you should take before creating this file:

1. Use the [`datasets-tagging` application](https://huggingface.co/datasets/tagging/) to create metadata tags in YAML format. These tags are used for a variety of search features on the Hugging Face Hub and ensure your dataset can be easily found by members of the community. Since we have created a custom dataset here, you'll need to clone the `datasets-tagging` repository and run the application locally. Here's what the interface looks like:

<img style="float=center;" width="70%" src="sections/section_5/images/dataset_card_assistant.png">

1. Read the [🤗 Datasets guide](https://github.com/huggingface/datasets/blob/master/templates/README_guide.md) on creating informative dataset cards and use it as a template.

You can create the *README.md* file directly on the Hub, and you can find a template dataset card in the `lewtun/github-issues` dataset repository. A screenshot of the filled-out dataset card is shown below.

<img style="float=center;" width="70%" src="sections/section_5/images/lewtun_github_issues.png">

> ✏️ Try it out! <font color="darkgreen">Use the `dataset-tagging` application and [🤗 Datasets guide](https://github.com/huggingface/datasets/blob/master/templates/README_guide.md) to complete the *README.md* file for your GitHub issues dataset.</font>

In [ ]:
# Trying it out
print("Procedure")
print('1. Read the above section on "Creating a dataset card".')
text_2 = "https://huggingface.co/spaces/huggingface/datasets-tagging"
print(f'2. Create tags with the "HuggingFace Dataset Tagger" at {text_2}.')
text_3 = "https://huggingface.co/datasets/card-creator"
print(f'3. Create a README.md file / dataset card with the "card-creator" at {text_3}.')
text_4 = "https://huggingface.co/datasets/mdroth/github_issues_10"
print(f'4. Upload the README.md file / dataset card to the model repo (in this case {text_4}):')
print('=> "Files and versions" > "Add file" > "Upload file" => Upload the README.md file')
text_5_1 = "https://huggingface.co/docs/datasets/v2.0.0/dataset_card"
text_5_2 = "https://huggingface.co/datasets/mdroth/github_issues_10"
print(f'\nSee {text_5_1} for a guide and {text_5_2} for an example.')

That's it! We've seen in this section that creating a good dataset can be quite involved, but fortunately uploading it and sharing it with the community is not. In the next section we'll use our new dataset to create a semantic search engine with 🤗 Datasets that can match questions to the most relevant issues and comments.

> ✏️ Try it out! <font color="darkgreen">Go through the steps we took in this section to create a dataset of GitHub issues for your favorite open source library (pick something other than 🤗 Datasets, of course!). For bonus points, fine-tune a multilabel classifier to predict the tags present in the `labels` field.</font>

In [ ]:
# Trying it out
import json
from datasets import DatasetDict
## load the file "transformers-issues.jsonl" that has been created by using 'fetch_issues(repo="transformers")' ...
## ... instead of 'fetch_issues()' just below the definition of the 'fetch_issues()' function further above
fetch_issues(repo="transformers")

In [ ]:
transformers_issues_dataset = transformers_issues_dataset.select_columns(["body", "labels"])
transformers_issues_dataset = transformers_issues_dataset.filter(lambda x: x["body"]!="" and x["labels"]!=[])
transformers_issues_dataset

In [ ]:
transformers_issues_dataset.filter(lambda x: x["body"]!=None and x["labels"]!=[])

In [ ]:
transformers_issues_dataset.filter(lambda x: x["body"]!=None and len(x["body"])>=100 and len(x["labels"])>=1)

In [ ]:
closed_dataset.filter(lambda x: x["pull_request"] is not None)

In [ ]:
transformers_issues_dataset["body"][:3]

In [ ]:
print(transformers_issues_dataset["labels"][7])

In [ ]:
for i, instance in enumerate(transformers_issues_dataset[:10]):
    print(f"\n\n#####\n# {i} #\n#####\n\nLabel:\t{instance}")

In [ ]:
transformers_issues_dataset["labels"][:10]

In [ ]:
transformers_issues_dataset[:10].filter(lambda item: item["labels"]!=None)

In [ ]:
transformer_issue_labels_dataset = 

In [ ]:
transformers_issue_reactions_dataset = transformers_issues_dataset.filter(lambda item: item["reactions"]["total_count"]>0)
transformers_issue_reactions_dataset

In [ ]:
for instance_i in transformers_issues_dataset[5]:
    print(instance_i)

In [ ]:
transformers_issues_dataset["reactions"][0:150]

In [ ]:
for issue_i in transformers_issues_dataset[0:3]:
    if len(issue_i["body"]) > 50 and len(issue_i["body"]) < 1500 and issue_i

In [ ]:
for i in range(3):
    if transformer
    print(transformers_issues_dataset[i])#["labels"])#["name"]

In [ ]:
transformers_issues_dataset["labels"] # for each instance, get the "name" values for the following keys "name" and 

In [ ]:
for label in transformers_issues_dataset["labels"]:#[:5]: # candidates: "reactions"
    if label!=[]:
        print(label[0]["name"])

In [ ]:
transformers_issues_text_dataset_1 = transformers_issues_dataset.rename_column(
    original_column_name="repository_url", new_column_name="text"
)
transformers_issues_text_dataset_0 = transformers_issues_text_dataset_1.rename_column(
    original_column_name="labels_url", new_column_name="num_labels"
)
transformers_issues_text_dataset = transformers_issues_text_dataset_0.rename_column(
    original_column_name="comments_url", new_column_name="arr_labels"
)
del transformers_issues_dataset, transformers_issues_text_dataset_0, transformers_issues_text_dataset_1
## combine the columns "title", "comments", "reactions", and "body" into a single string in the new "text" column
feature_keys = ["title", "comments", "reactions", "body"]
reaction_keys = ["+1", "-1", "laugh", "hooray", "heart", "rocket", "eyes"]
def make_text(item):
    text = ""
    for fk_i in feature_keys:
        if fk_i=="reactions":
            text += f"\n\n{fk_i.upper()}"
            reactions = item[fk_i]
            reactions_json = json.loads(json.dumps(reactions, indent = 4))
            for rk_i in reaction_keys:
                rk_iCount = reactions_json[rk_i]
                text += f"\n{rk_i}: {rk_iCount}"
        else:
            text += f"\n\n{fk_i.upper()}\n{item[fk_i]}"
    item["text"] = text
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_text)
## build labels (list of strings)
def make_labels(item):
    labels = item["labels"]
    label_list = []
    for label in labels:
        label_json = json.loads(json.dumps(label, indent=4))
        label_name = label_json["name"]
        label_list.append(label_name)
    item["labels"] = label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_labels)
## build num_labels (list of numbers) ...
## ... see also https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-a-list-of-lists
flat_label_list = [label for labels_i in transformers_issues_text_dataset["labels"] for label in labels_i]
unique_labels = list(set(flat_label_list))
n_unique_labels = len(unique_labels)
def make_num_labels(item):
    label_list = item["labels"]
    num_label_list = []
    for label in label_list:
        num_label = unique_labels.index(label)
        num_label_list.append(num_label)
    item["num_labels"] = num_label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_num_labels)
## build arr_labels (full list of 0s and 1s)
def make_arr_labels(item):
    num_labels = item["num_labels"]
    arr_label_list = [0 for _ in range(n_unique_labels)]
    for num_label in num_labels:
        arr_label_list[num_label] = 1
    item["arr_labels"] = arr_label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_arr_labels)
## filter for instances with labels!=[]
idx = 0
print(f'Before filtering\n=> Empty labels for index {idx}:\t\t{transformers_issues_text_dataset["labels"][idx]}')
transformers_issues_text_dataset = transformers_issues_text_dataset.filter(lambda x: x["labels"]!=[])
print(f'After filtering\n=> Non-empty labels for index {idx}:\t{transformers_issues_text_dataset["labels"][idx]}')
## remove all columns but "labels", "text", and "url"
keep_keys = ["text", "labels", "num_labels", "arr_labels", "url"]
remove_keys = [key for key in list(transformers_issues_text_dataset.features.keys()) if key not in keep_keys]
transformers_issues_text_dataset = transformers_issues_text_dataset.remove_columns(remove_keys)
## show dataset
print(f"\nDataset with num_labels:\n{transformers_issues_text_dataset}")
idx = 5 # 5 or 13 or (almost) anything
print(f'\n{3*"#"+" "}text{" "+71*"#"}{transformers_issues_text_dataset["text"][idx]}')
print(f'\n{3*"#"+" "}labels{" "+69*"#"}\n\n{transformers_issues_text_dataset["labels"][idx]}')
print(f'\n{3*"#"+" "}num_labels{" "+65*"#"}\n\n{transformers_issues_text_dataset["num_labels"][idx]}')
print(f'\n{3*"#"+" "}arr_labels{" "+65*"#"}\n\n{transformers_issues_text_dataset["arr_labels"][idx]}')
print(f'\n{3*"#"+" "}url{" "+72*"#"}\n\n{transformers_issues_text_dataset["url"][idx]}')
print(f'\n{80*"#"}\n')
## DatasetDict: make splits, build, print, and push to hub
train_dev = transformers_issues_text_dataset.train_test_split(shuffle=True, seed=42, test_size=0.003)
train_validTest = train_dev["train"].train_test_split(shuffle=True, seed=42, test_size=0.36)
valid_test = train_validTest["test"].train_test_split(shuffle=True, seed=42, test_size=5/9)
transformers_issues_text_dataset = DatasetDict({
    "train": train_validTest["train"],
    "valid": valid_test["train"],
    "test": valid_test["test"],
    "dev": train_dev["test"]
})
repo_id="transformers_issues_labels"
transformers_issues_text_dataset.save_to_disk(f"sections/section_5/data/GitHub-repo-{repo_id}") # save locally
transformers_issues_text_dataset.push_to_hub(repo_id=repo_id) # push to hub
transformers_issues_text_dataset

In [ ]:
transformers_issues_text_dataset

In [ ]:
# useful keys: "text", "num_labels"
# useless keys: "url", "arr_labels"
transformers_issues_text_dataset["train"]["labels"]

In [ ]:
transformers_issues_text_dataset["dev"]["arr_labels"]

In [ ]:
for key in list(transformers_issues_text_dataset["dev"].features.keys()):
    print(f"\n{key}")
    print(f"{transformers_issues_text_dataset["dev"][key]}")

In [ ]:
# still trying it out
## imports
from transformers import DataCollatorWithPadding, BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
## define custom trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs["token_type_ids"]
        )
        loss = torch.nn.BCEWithLogitsLoss()(outputs["logits"].float(), inputs["labels"].float())
        return (loss, outputs) if return_outputs else loss
## load dataset, tokenize, adapt columns, and apply datacollator
checkpoint = "bert-base-cased"
transformers_tokenizer = BertTokenizer.from_pretrained(checkpoint)
def transformers_tokenize_function(item):
    return transformers_tokenizer(item["text"], padding=True, truncation=True)
transformers_tokenized_datasets = (
    load_dataset("mdroth/transformers_issues_labels")
    .map(transformers_tokenize_function, batched=True)
    .remove_columns(column_names=["url", "text", "num_labels", "labels"])
    .rename_column("arr_labels", "labels")
)
transformers_data_collator = DataCollatorWithPadding(tokenizer=transformers_tokenizer)
## training arguments
training_args = TrainingArguments(
    "sections/section_5/logs/transformers_issues_model",
    load_best_model_at_end=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4
)
## model
transformers_model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=n_unique_labels)
## trainer
trainer = CustomTrainer(
    transformers_model,
    training_args,
    train_dataset=transformers_tokenized_datasets["dev"],
    eval_dataset=transformers_tokenized_datasets["dev"],
    data_collator=transformers_data_collator,
    tokenizer=transformers_tokenizer
)
## train
trainer.train()

In [ ]:
# Trying it out
import json
from datasets import DatasetDict
## load the file "transformers-issues.jsonl" that has been created by using 'fetch_issues(repo="transformers")' ...
## ... instead of 'fetch_issues()' just below the definition of the 'fetch_issues()' function further above
fetch_issues(repo="transformers")
transformers_issues_dataset = load_dataset("json", data_files="sections/section_5/data/transformers-issues.jsonl", split="train")
## add columns "text" and "num_labels"
print(transformers_issues_dataset)
transformers_issues_text_dataset_1 = transformers_issues_dataset.rename_column(
    original_column_name="repository_url", new_column_name="text"
)
transformers_issues_text_dataset_0 = transformers_issues_text_dataset_1.rename_column(
    original_column_name="labels_url", new_column_name="num_labels"
)
transformers_issues_text_dataset = transformers_issues_text_dataset_0.rename_column(
    original_column_name="comments_url", new_column_name="arr_labels"
)
del transformers_issues_dataset, transformers_issues_text_dataset_0, transformers_issues_text_dataset_1
## combine the columns "title", "comments", "reactions", and "body" into a single string in the new "text" column
feature_keys = ["title", "comments", "reactions", "body"]
reaction_keys = ["+1", "-1", "laugh", "hooray", "heart", "rocket", "eyes"]
def make_text(item):
    text = ""
    for fk_i in feature_keys:
        if fk_i=="reactions":
            text += f"\n\n{fk_i.upper()}"
            reactions = item[fk_i]
            reactions_json = json.loads(json.dumps(reactions, indent = 4))
            for rk_i in reaction_keys:
                rk_iCount = reactions_json[rk_i]
                text += f"\n{rk_i}: {rk_iCount}"
        else:
            text += f"\n\n{fk_i.upper()}\n{item[fk_i]}"
    item["text"] = text
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_text)
## build labels (list of strings)
def make_labels(item):
    labels = item["labels"]
    label_list = []
    for label in labels:
        label_json = json.loads(json.dumps(label, indent=4))
        label_name = label_json["name"]
        label_list.append(label_name)
    item["labels"] = label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_labels)
## build num_labels (list of numbers) ...
## ... see also https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-a-list-of-lists
flat_label_list = [label for labels_i in transformers_issues_text_dataset["labels"] for label in labels_i]
unique_labels = list(set(flat_label_list))
n_unique_labels = len(unique_labels)
def make_num_labels(item):
    label_list = item["labels"]
    num_label_list = []
    for label in label_list:
        num_label = unique_labels.index(label)
        num_label_list.append(num_label)
    item["num_labels"] = num_label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_num_labels)
## build arr_labels (full list of 0s and 1s)
def make_arr_labels(item):
    num_labels = item["num_labels"]
    arr_label_list = [0 for _ in range(n_unique_labels)]
    for num_label in num_labels:
        arr_label_list[num_label] = 1
    item["arr_labels"] = arr_label_list
    return item
transformers_issues_text_dataset = transformers_issues_text_dataset.map(make_arr_labels)
## filter for instances with labels!=[]
idx = 0
print(f'Before filtering\n=> Empty labels for index {idx}:\t\t{transformers_issues_text_dataset["labels"][idx]}')
transformers_issues_text_dataset = transformers_issues_text_dataset.filter(lambda x: x["labels"]!=[])
print(f'After filtering\n=> Non-empty labels for index {idx}:\t{transformers_issues_text_dataset["labels"][idx]}')
## remove all columns but "labels", "text", and "url"
keep_keys = ["text", "labels", "num_labels", "arr_labels", "url"]
remove_keys = [key for key in list(transformers_issues_text_dataset.features.keys()) if key not in keep_keys]
transformers_issues_text_dataset = transformers_issues_text_dataset.remove_columns(remove_keys)
## show dataset
print(f"\nDataset with num_labels:\n{transformers_issues_text_dataset}")
idx = 5 # 5 or 13 or (almost) anything
print(f'\n{3*"#"+" "}text{" "+71*"#"}{transformers_issues_text_dataset["text"][idx]}')
print(f'\n{3*"#"+" "}labels{" "+69*"#"}\n\n{transformers_issues_text_dataset["labels"][idx]}')
print(f'\n{3*"#"+" "}num_labels{" "+65*"#"}\n\n{transformers_issues_text_dataset["num_labels"][idx]}')
print(f'\n{3*"#"+" "}arr_labels{" "+65*"#"}\n\n{transformers_issues_text_dataset["arr_labels"][idx]}')
print(f'\n{3*"#"+" "}url{" "+72*"#"}\n\n{transformers_issues_text_dataset["url"][idx]}')
print(f'\n{80*"#"}\n')
## DatasetDict: make splits, build, print, and push to hub
train_dev = transformers_issues_text_dataset.train_test_split(shuffle=True, seed=421, test_size=0.003)
train_validTest = train_dev["train"].train_test_split(shuffle=True, seed=42, test_size=0.36)
valid_test = train_validTest["test"].train_test_split(shuffle=True, seed=42, test_size=5/9)
transformers_issues_text_dataset = DatasetDict({
    "train": train_validTest["train"],
    "valid": valid_test["train"],
    "test": valid_test["test"],
    "dev": train_dev["test"]
})
print(transformers_issues_text_dataset)
repo_id="transformers_issues_labels"
transformers_issues_text_dataset.save_to_disk(f"sections/section_5/data/GitHub-repo-{repo_id}") # save locally
transformers_issues_text_dataset.push_to_hub(repo_id=repo_id) # save on hub

For *Try it out!*-**bonus points**, the following cell builds and trains a multilabel classifier for the (multilabel) labels of the dataset that has just been created. See also [this discussion on the HuggingFace forum](https://discuss.huggingface.co/t/reshaping-logits-when-using-trainer/18214?u=mdroth).

In [ ]:
from transformers import BertConfig, BertForSequenceClassification

num_labels = 82  # Set to the correct number of labels in your dataset.
config = BertConfig.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="multi_label_classification"  # This sets up the model to use BCEWithLogitsLoss.
)
model = BertForSequenceClassification.from_pretrained(checkpoint, config=config)




#from transformers import BertConfig, BertForSequenceClassification, BertTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments
from datasets import load_dataset

# Define checkpoint and load tokenizer
checkpoint = "bert-base-cased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)

# Prepare your dataset (example using your dataset)
def tokenize_function(item):
    return tokenizer(item["text"], padding=True, truncation=True)

tokenized_datasets = (
    load_dataset("mdroth/transformers_issues_labels")
    .map(tokenize_function, batched=True)
    .remove_columns(column_names=["url", "text", "num_labels", "labels"])
    .rename_column("arr_labels", "labels")
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Specify the number of unique labels
n_unique_labels = 82  # Replace with the correct number

# Create a model configuration with problem_type set to multi_label_classification.
config = BertConfig.from_pretrained(checkpoint,
                                    num_labels=n_unique_labels,
                                    problem_type="multi_label_classification")
model = BertForSequenceClassification.from_pretrained(checkpoint, config=config)

# Define training arguments.
training_args = TrainingArguments(
    output_dir="sections/section_5/logs/transformers_issues_model",
    report_to=[],  # disable external logging
    load_best_model_at_end=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
)

# Initialize the Trainer without any custom code.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["dev"],
    eval_dataset=tokenized_datasets["dev"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Start training.
trainer.train()

In [ ]:
# still trying it out
## imports
from transformers import DataCollatorWithPadding, BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
## define custom trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs["token_type_ids"]
        )
        loss = torch.nn.BCEWithLogitsLoss()(outputs["logits"].float(), inputs["labels"].float())
        return (loss, outputs) if return_outputs else loss
## load dataset, tokenize, adapt columns, and apply datacollator
checkpoint = "bert-base-cased"
transformers_tokenizer = BertTokenizer.from_pretrained(checkpoint)
def transformers_tokenize_function(item):
    return transformers_tokenizer(item["text"], padding=True, truncation=True)
transformers_tokenized_datasets = (
    load_dataset("mdroth/transformers_issues_labels")
    .map(transformers_tokenize_function, batched=True)
    .remove_columns(column_names=["url", "text", "num_labels", "labels"])
    .rename_column("arr_labels", "labels")
)
transformers_data_collator = DataCollatorWithPadding(tokenizer=transformers_tokenizer)
## training arguments
training_args = TrainingArguments(
    "sections/section_5/logs/transformers_issues_model",
    report_to=[],
    load_best_model_at_end=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4
)
## model
transformers_model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=n_unique_labels)
## trainer
trainer = CustomTrainer(
    transformers_model,
    training_args,
    train_dataset=transformers_tokenized_datasets["dev"],
    eval_dataset=transformers_tokenized_datasets["dev"],
    data_collator=transformers_data_collator,
    tokenizer=transformers_tokenizer
)
## train
trainer.train()

**Restart the kernel in preparation for the next section.**

In [ ]:
import os
os._exit(00)

## [Semantic search with FAISS](https://huggingface.co/course/chapter5/6?fw=pt)

In [section 5](https://huggingface.co/course/chapter5/5), we created a dataset of GitHub issues and comments from the 🤗 Datasets repository. In this section we'll use this information to build a search engine that can help us find answers to our most pressing questions about the library!

In [ ]:
from IPython.display import HTML
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/OATCgQtNX2o" allowfullscreen></iframe>')

### Using embeddings for semantic search
As we saw in [Chapter 1](https://huggingface.co/course/chapter1), Transformer-based language models represent each token in a span of text as an *embedding vector*. It turns out that one can "pool" the individual embeddings to create a vector representation for whole sentences, paragraphs, or (in some cases) documents. These embeddings can then be used to find similar documents in the corpus by computing the dot-product similarity (or some other similarity metric) between each embedding and returning the documents with the greatest overlap.

In this section we'll use embeddings to develop a semantic search engine. These search engines offer several advantages over conventional approaches that are based on matching keywords in a query with the documents.

<img style="float=center;" width="30%" src="sections/section_5/images/relevant_doc.png">

### Loading and preparing the dataset
The first thing we need to do is download our dataset of GitHub issues, so let's use the 🤗 Hub library to resolve the URL where our file is stored on the Hugging Face Hub:

In [ ]:
from huggingface_hub import hf_hub_url
data_files = hf_hub_url(
    repo_id="lewtun/github-issues",
    filename="datasets-issues-with-comments.jsonl",
    repo_type="dataset"
)
data_files

With the URL stored in data_files, we can then load the remote dataset using the method introduced in [section 2](https://huggingface.co/course/chapter5/2):

In [ ]:
from datasets import load_dataset
issues_dataset = load_dataset("json", data_files=data_files, split="train")
issues_dataset

Here we've specified the default train split in `load_dataset()`, so it returns a `Dataset` instead of a `DatasetDict`. The first order of business is to filter out the pull requests, as these tend to be rarely used for answering user queries and will introduce noise in our search engine. As should be familiar by now, we can use the `Dataset.filter()` function to exclude these rows in our dataset. While we're at it, let's also filter out rows with no comments, since these provide no answers to user queries:

In [ ]:
issues_dataset = issues_dataset.filter(lambda x: (x["is_pull_request"]==False and len(x["comments"]) > 0))
issues_dataset

We can see that there are a lot of columns in our dataset, most of which we don't need to build our search engine. From a search perspective, the most informative columns are `title`, `body`, and `comments`, while `html_url` provides us with a link back to the source issue. Let's use the `Dataset.remove_columns()` function to drop the rest:

In [ ]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

To create our embeddings we'll augment each comment with the issue's title and body, since these fields often include useful contextual information. Because our comments column is currently a list of comments for each issue, we need to "explode" the column so that each row consists of an (`html_url`, `title`, `body`, `comment`) tuple. In Pandas, we can do this with the [`DataFrame.explode()` function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.explode.html), which creates a new row for each element in a list-like column, while replicating all the other column values. To see this in action, let's first switch to the Pandas `DataFrame` format:

In [ ]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]
df

If we inspect the first row in this `DataFrame` we can see there are two comments associated with this issue:

In [ ]:
print(df["comments"][0].tolist())
len(df["comments"][0].tolist())

When we explode `df`, we expect to get one row for each of these comments. Let's check if that's the case:

In [ ]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(9)

Great, we can see the rows have been replicated, with the `comments` column containing the individual comments! Now that we're finished with Pandas, we can quickly switch back to a `Dataset` by loading the `DataFrame` in memory:

In [ ]:
from datasets import Dataset
comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Okay, this has given us a few thousand comments to work with!
> ✏️ Try it out! <font color="darkgreen">See if you can use `Dataset.map()` to explode the `comments` column of `issues_dataset` *without* resorting to the use of Pandas. This is a little tricky; you might find the ["Batch mapping"](https://huggingface.co/docs/datasets/v2.0.0/about_map_batch?batch-mapping#batch-mapping) section of the 🤗 Datasets documentation useful for this task.</font>

In [ ]:
# Trying it out
def explode_comments(items):
    # initialize empty arrays "html_url_arr = []", "title_arr = []", "comments_arr = []", and "body_arr = []"
    # loop over items
    # for each item:
    ## get the values for "html_url", "title", and "body"
    ## get "n_i" = the number of copies that need to be made (length of this item's "comments_arr" array)
    ## loop index "ii" over "range(n_i)"
    ### in step ii of the loop, get the ii-th value (= i-th comment) of the current item's "comments_arr" array
    ### append all current values (html_url, title, comment, body) to their corresponding arrays
    # build the dictionary and return it
    html_url_arr = []
    title_arr = []
    comments_arr = [] # only nested array
    body_arr = []
    n_items = len(items["html_url"])
    for i in range(n_items):
        html_url = items["html_url"][i]
        title = items["title"][i]
        body = items["body"][i]
        n_i = len(items["comments"][i])
        for ii in range(n_i):
            comment = items["comments"][i][ii]
            html_url_arr.append(html_url)
            title_arr.append(title)
            comments_arr.append(comment)
            body_arr.append(body)
    return {"html_url": html_url_arr, "title": title_arr, "comments": comments_arr, "body": body_arr}
# https://huggingface.co/docs/datasets/v2.0.0/about_map_batch?batch-mapping#batch-mapping
try_issues_dataset_exploded_comments = issues_dataset.map(explode_comments, batched=True)
# https://discuss.huggingface.co/t/how-do-you-rename-a-column-in-a-dataset/15121
try_issues_dataset_exploded_comments = try_issues_dataset_exploded_comments.rename_column("comments", "comment")
print("try_issues_dataset_exploded_comments")
try_issues_dataset_exploded_comments.set_format("pandas")
try_issues_dataset_exploded_comments[:]

Now that we have one comment per row, let's create a new `comments_length` column that contains the number of words per comment:

In [ ]:
comments_dataset = comments_dataset.map(lambda x: {"comment_length": len(x["comments"].split())})

We can use this new column to filter out short comments, which typically include things like "cc @lewtun" or "Thanks!" that are not relevant for our search engine. There's no precise number to select for the filter, but around 15 words seems like a good start:

In [ ]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Having cleaned up our dataset a bit, let's concatenate the issue title, description, and comments together in a new `text` column. As usual, we'll write a simple function that we can pass to `Dataset.map()`:

In [ ]:
def concatenate_text(examples):
    return {"text": examples["title"] + " \n " + examples["body"] + " \n " + examples["comments"]}
comments_dataset = comments_dataset.map(concatenate_text)

We're finally ready to create some embeddings! Let's take a look.

### Creating text embeddings
We saw in [Chapter 2](https://huggingface.co/course/chapter2) that we can obtain token embeddings by using the `AutoModel` class. All we need to do is pick a suitable checkpoint to load the model from. Fortunately, there's a library called `sentence-transformers` that is dedicated to creating embeddings. As described in the library's [documentation](https://www.sbert.net/examples/applications/semantic-search/README.html#symmetric-vs-asymmetric-semantic-search), our use case is an example of *asymmetric semantic search* because we have a short query whose answer we'd like to find in a longer document, like a an issue comment. The handy [model overview table](https://www.sbert.net/docs/pretrained_models.html#model-overview) in the documentation indicates that the `multi-qa-mpnet-base-dot-v1` checkpoint has the best performance for semantic search, so we'll use that for our application. We'll also load the tokenizer using the same checkpoint:

In [ ]:
from transformers import AutoTokenizer, AutoModel
model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

To speed up the embedding process, it helps to place the model and inputs on a GPU device, so let's do that now:

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # use gpu if possible, else cpu
print(device)
model.to(device)

As we mentioned earlier, we'd like to represent each entry in our GitHub issues corpus as a single vector, so we need to "pool" or average our token embeddings in some way. One popular approach is to perform *CLS pooling* on our model's outputs, where we simply collect the last hidden state for the special `[CLS]` token. The following function does the trick for us:

In [ ]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

Next, we'll create a helper function that will tokenize a list of documents, place the tensors on the GPU, feed them to the model, and finally apply CLS pooling to the outputs:

In [ ]:
def get_embeddings(text_list):
    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

We can test the function works by feeding it the first text entry in our corpus and inspecting the output shape:

In [ ]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

Great, we've converted the first entry in our corpus into a 768-dimensional vector! We can use `Dataset.map()` to apply our `get_embeddings()` function to each row in our corpus, so let's create a new `embeddings` column as follows:

In [ ]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Notice that we've converted the embeddings to NumPy arrays — that's because 🤗 Datasets requires this format when we try to index them with FAISS, which we'll do next.

### Using FAISS for efficient similarity search
Now that we have a dataset of embeddings, we need some way to search over them. To do this, we'll use a special data structure in 🤗 Datasets called a *FAISS index*. [FAISS](https://faiss.ai/) (short for Facebook AI Similarity Search) is a library that provides efficient algorithms to quickly search and cluster embedding vectors.

The basic idea behind FAISS is to create a special data structure called an *index* that allows one to find which embeddings are similar to an input embedding. Creating a FAISS index in 🤗 Datasets is simple — we use the `Dataset.add_faiss_index()` function and specify which column of our dataset we'd like to index:

In [ ]:
embeddings_dataset.add_faiss_index(column="embeddings")

We can now perform queries on this index by doing a nearest neighbor lookup with the `Dataset.get_nearest_examples()` function. Let's test this out by first embedding a question as follows:

In [ ]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

Just like with the documents, we now have a 768-dimensional vector representing the query, which we can compare against the whole corpus to find the most similar embeddings:

In [ ]:
scores, samples = embeddings_dataset.get_nearest_examples("embeddings", question_embedding, k=5)

The `Dataset.get_nearest_examples()` function returns a tuple of scores that rank the overlap between the query and the document, and a corresponding set of samples (here, the 5 best matches). Let's collect these in a `pandas.DataFrame` so we can easily sort them:

In [ ]:
import pandas as pd
samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

Now we can iterate over the first few rows to see how well our query matched the available comments:

In [ ]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

Not bad! Our second hit seems to match the query.
> ✏️ Try it out! <font color="darkgreen">Create your own query and see whether you can find an answer in the retrieved documents. You might have to increase the `k` parameter in `Dataset.get_nearest_examples()` to broaden the search.</font>

In [ ]:
# Trying it out
## query and embeddings
query = "How does batch mapping work?" # custom query
query_embedding = get_embeddings([query]).cpu().detach().numpy()
print(query_embedding.shape)
## sample the k nearest instances as well as their score (wrt the query)
scores, samples = embeddings_dataset.get_nearest_examples("embeddings", query_embedding, k=15) # maybe adapt k
## turn samples into a pandas dataframe, add the scores as a column, and sort the rows by their scores
samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)
## print the results sample by sample
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

## [🤗 Datasets, check!](https://huggingface.co/course/chapter5/7?fw=pt)
Well, that was quite a tour through the 🤗 Datasets library — congratulations on making it this far! With the knowledge that you've gained from this chapter, you should be able to:
- Load datasets from anywhere, be it the Hugging Face Hub, your laptop, or a remote server at your company.
- Wrangle your data using a mix of the `Dataset.map()` and `Dataset.filter()` functions.
- Quickly switch between data formats like Pandas and NumPy using `Dataset.set_format()`.
- Create your very own dataset and push it to the Hugging Face Hub.
- Embed your documents using a Transformer model and build a semantic search engine using FAISS.

In [Chapter 7](https://huggingface.co/course/chapter7), we'll put all of this to good use as we take a deep dive into the core NLP tasks that Transformer models are great for. Before jumping ahead, though, put your knowledge of 🤗 Datasets to the test with a quick quiz!

## [End-of-chapter quiz](https://huggingface.co/course/chapter5/8?fw=pt)
This chapter covered a lot of ground! Don't worry if you didn't grasp all the details; the next chapters will help you understand how things work under the hood.

Before moving on, though, let's test what you learned in this chapter.

**1. The `load_dataset()` function in 🤗 Datasets allows you to load a dataset from which of the following locations?**<br>
⚫️ Locally, e.g. on your laptop
> **Correct!** Correct! You can pass the paths of local files to the `data_files` argument of `load_dataset()` to load local datasets.

⚫️ The Hugging Face Hub
> **Correct!** Correct! You can load datasets on the Hub by providing the dataset ID, e.g. `load_dataset('emotion')`.

⚫️ A remote server
> **Correct!** Correct! You can pass URLs to the `data_files` argument of `load_dataset()` to load remote files.

**2. Suppose you load one of the GLUE tasks as follows:**
```python
from datasets import load_dataset
dataset = load_dataset("glue", "mrpc", split="train")
```
Which of the following commands will produce a random sample of 50 elements from dataset?<br>
⚪️ `dataset.sample(50)`<br>
⚫️ `dataset.shuffle().select(range(50))`
> **Correct!** Correct! As you saw in this chapter, you first shuffle the dataset and then select the samples from it.

⚪️ `dataset.select(range(50)).shuffle()`

**3. Suppose you have a dataset about household pets called `pets_dataset`, which has a `name` column that denotes the name of each pet. Which of the following approaches would allow you to filter the dataset for all pets whose names start with the letter "L"?**<br>
⚫️ `pets_dataset.filter(lambda x : x['name'].startswith('L'))`
> **Correct!** Correct! Using a Python lambda function for these quick filters is a great idea. Can you think of another solution?

⚪️ `pets_dataset.filter(lambda x['name'].startswith('L'))`<br>
⚫️ Create a function like `def filter_names(x): return x['name'].startswith('L')` and run `pets_dataset.filter(filter_names)`.
> **Correct!** Correct! Just like with `Dataset.map()`, you can pass explicit functions to `Dataset.filter()`. This is useful when you have some complex logic that isn't suitable for a short lambda function. Which of the other solutions would work?

**4. What is memory mapping?**<br>
⚪️ A mapping between CPU and GPU RAM<br>
⚫️ A mapping between RAM and filesystem storage
> **Correct!** Correct! 🤗 Datasets treats each dataset as a memory-mapped file. This allows the library to access and operate on elements of the dataset without needing to fully load it into memory.

⚪️ A mapping between two files in the 🤗 Datasets cache

**5. Which of the following are the main benefits of memory mapping?**<br>
⚫️ Accessing memory-mapped files is faster than reading from or writing to disk.
> **Correct!** Correct! This allows 🤗 Datasets to be blazing fast. That's not the only benefit, though.

⚫️ Applications can access segments of data in an extremely large file without having to read the whole file into RAM first.
> **Correct!** Correct! This allows 🤗 Datasets to load multi-gigabyte datasets on your laptop without blowing up your CPU. What other advantage does memory mapping offer?

⚪️ It consumes less energy, so your battery lasts longer.

**6. Why does the following code fail?**
```python
from datasets import load_dataset
dataset = load_dataset("allocine", streaming=True, split="train")
dataset[0]
```
<br>
⚪️ It tries to stream a dataset that's too large to fit in RAM.<br>
⚫️ It tries to access an `IterableDataset`.
> **Correct!** Correct! An `IterableDataset` is a generator, not a container, so you should access its elements using `next(iter(dataset))`.

⚪️ The `allocine` dataset doesn't have a `train` split.

**7. Which of the following are the main benefits of creating a dataset card?**<br>
⚫️ It provides information about the intended use and supported tasks of the dataset so others in the community can make an informed decision about using it.
> **Correct!** Undocumented datasets may be used to train models that may not reflect the intentions of the dataset creators, or may produce models whose legal status is murky if they're trained on data that violates privacy or licensing restrictions. This isn't the only benefit, though!

⚫️ It helps draw attention to the biases that are present in a corpus.
> **Correct!** Correct! Almost all datasets have some form of bias, which can produce negative consequences downstream. Being aware of them helps model builders understand how to address the inherent biases. What else do dataset cards help with?

⚫️ It improves the chances that others in the community will use my dataset.
> **Correct!** Correct! A well-written dataset card will tend to lead to higher usage of your precious dataset. What other benefits does it offer?

**8. What is semantic search?**<br>
⚪️ A way to search for exact matches between the words in a query and the documents in a corpus<br>
⚫️ A way to search for matching documents by understanding the contextual meaning of a query
> **Correct!** Correct! Semantic search uses embedding vectors to represent queries and documents, and uses a similarity metric to measure the amount of overlap between them. How else might you describe it?

⚫️ A way to improve search accuracy<br>
> **Correct!** Correct! Semantic search engines can capture the intent of a query much better than keyword matching and typically retrieve documents with higher precision. But this isn't the only right answer - what else does semantic search provide?

**9. For asymmetric semantic search, you usually have:**<br>
⚫️ A short query and a longer paragraph that answers the query
> **Correct!** Correct!

⚪️ Queries and paragraphs that are of about the same length<br>
⚪️ A long query and a shorter paragraph that answers the query


**10. Can I use 🤗 Datasets to load data for use in other domains, like speech processing?**<br>
⚪️ No<br>
⚫️ Yes
> **Correct!** Correct! Check out the exciting developments with speech and vision in the 🤗 Transformers library to see how 🤗 Datasets is used in these domains.